# 03 — Pipeline Completo del Router MoE (Pasos A → M)

Este notebook implementa el **pipeline completo** del Vision Router para el sistema MoE,
siguiendo exactamente el diagrama propuesto por el profesor:

```
A → Datos crudos (5 datasets)
B → DataLoader Mixto
C → Collate Personalizado (separa 2D / 3D)
D,E → Patch Embedding (2D: 16×16 | 3D: 8×8×8)
F,G → Proyección Lineal compartida → 192 dim
H → Padding de secuencia → tensor homogéneo [B, 513, 192]
I → Token CLS + Positional Embeddings
J → 12 Bloques de Self-Attention
K → Extracción del Token CLS Final
L → Vector de Representación Global: 1×192
M → Capa Lineal del Router: 192 → 5
```

**Nota técnica clave:** El `SwitchablePatchEmbed` usa *corrimiento de ventana*:
no es un modelo, es un **método** que convierte imágenes/volúmenes en valores
característicos de dimensión homogénea `1×192`, sin importar si la entrada es 2D o 3D.

Al final, los CLS tokens se guardan en disco para el **Estudio de Ablación** posterior
(Linear vs GMM vs Naive Bayes vs k-NN).


## Fase 0: Entorno y Configuración (Punto A)
Instalamos dependencias, montamos Drive y configuramos rutas.


In [2]:
!pip install -q SimpleITK nibabel torch torchvision monai einops timm
from google.colab import drive
import os, time, shutil, glob, zipfile, subprocess
from pathlib import Path
from collections import Counter
import numpy as np
import cv2
from PIL import Image
import SimpleITK as sitk
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from monai.networks.blocks import PatchEmbed
import random
from torch.cuda.amp import autocast
from sklearn.model_selection import train_test_split
import timm

# ---------- Montar Google Drive ----------
drive.mount('/content/drive')

# ---------- Rutas ----------
RAW_DIR    = '/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/'
LOCAL_DEST = '/content/datasets/'
PROC_BASE  = '/content/drive/MyDrive/PROYECTO_MOE_VISION/02_Data_Processed/'
# Pesos: expertos, cabeza del router, artefactos de ablation (misma raíz en Drive)
WEIGHTS_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/'
FEATURE_DIR = WEIGHTS_DIR

os.makedirs(LOCAL_DEST, exist_ok=True)
os.makedirs(PROC_BASE, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# ---------- Mapeo de datasets (consigna) ----------
DATASET_ROOTS = {
    'NIH':      (os.path.join(LOCAL_DEST, 'NIH Chest X ray 14'),                0),
    'ISIC':     (os.path.join(LOCAL_DEST, 'ISIC 2019'),                         1),
    'Osteo':    (os.path.join(LOCAL_DEST, 'Knee Osteoarthritis Classification'), 2),
    'LUNA16':   (os.path.join(LOCAL_DEST, 'Luna16 Lung Cancer Dataset'),         3),
    'Pancreas': (os.path.join(LOCAL_DEST, 'Pancreas Cancer'),                    4),
}

print('Rutas configuradas.')
print(f'  RAW_DIR:      {RAW_DIR}')
print(f'  LOCAL_DEST:   {LOCAL_DEST}')
print(f'  WEIGHTS_DIR:  {WEIGHTS_DIR}')


Mounted at /content/drive
Rutas configuradas.
  RAW_DIR:      /content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/
  LOCAL_DEST:   /content/datasets/
  WEIGHTS_DIR:  /content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/


In [3]:
# =============================================================================
# Arquitecturas de los 5 expertos (embebido en el notebook; sin .py externo)
# =============================================================================

import os
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import timm
import torchvision.models as models
from torchvision.models.video import R3D_18_Weights, r3d_18


# ============================================================================
# Experto 1 (NIH) - Swin-Tiny (NIH_ChestXray_Swin_Tiny_Training.ipynb)
# 5 clases multietiqueta (Mass, Nodule, Effusion, Cardiomegaly, Pneumothorax).
# Pesos: suele guardarse como Experts_2D/MaxViT_NIH_5cls.pth (nombre historico en CONFIG).
# ============================================================================
class SwinNIHClassifier(nn.Module):
    """Misma envoltura que SwinClassifier en el notebook de entrenamiento (timm)."""

    def __init__(self, num_classes: int = 5, pretrained: bool = True):
        super().__init__()
        self.model = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


# ============================================================================
# Experto 3 (Osteo) - VGG16-BN
# ============================================================================
def build_vgg16_bn(num_classes: int = 5, pretrained: bool = True):
    model = models.vgg16_bn(weights="IMAGENET1K_V1" if pretrained else None)
    old_conv = model.features[0]
    new_conv = nn.Conv2d(1, 64, kernel_size=3, padding=1)
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        new_conv.bias.copy_(old_conv.bias)
    model.features[0] = new_conv
    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 512),
        nn.ReLU(True),
        nn.BatchNorm1d(512),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(True),
        nn.BatchNorm1d(256),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model


# ============================================================================
# Experto 4 (LUNA16 3D) - DCSwinBStyle3D (igual a LUNA16_Swin3D_Training.ipynb)
# Experto 5 (Pancreas 3D) - R3D18
# ============================================================================
from torch.utils.checkpoint import checkpoint as grad_checkpoint


class DCSwinBStyle3D(nn.Module):
    """Dual-branch estilo DCSwinB: Swin-Tiny 3D (inflate timm) + CNN 3D (stem+layer1+layer2 de R3D-18)."""

    def __init__(self, num_classes: int = 2, pretrained: bool = True, use_dwconv_mix: bool = True):
        super().__init__()
        self.use_dwconv_mix = use_dwconv_mix

        swin2d = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=0,
            img_size=256,
        )
        old_proj = swin2d.patch_embed.proj
        c_out = old_proj.out_channels

        self.patch_embed_3d = nn.Conv3d(1, c_out, kernel_size=(4, 4, 4), stride=(4, 4, 4), padding=0)
        w2d = old_proj.weight.data
        w_mean = w2d.mean(dim=1, keepdim=True)
        w3d = w_mean.unsqueeze(2).expand(-1, -1, 4, -1, -1) / 4.0
        with torch.no_grad():
            self.patch_embed_3d.weight.copy_(w3d)
            if old_proj.bias is not None and self.patch_embed_3d.bias is not None:
                self.patch_embed_3d.bias.copy_(old_proj.bias.data)

        if use_dwconv_mix:
            self.dw_mix = nn.Conv3d(c_out, c_out, 3, padding=1, groups=c_out, bias=True)
            nn.init.zeros_(self.dw_mix.weight)
            if self.dw_mix.bias is not None:
                nn.init.zeros_(self.dw_mix.bias)
        else:
            self.dw_mix = None

        self.patch_embed_norm = swin2d.patch_embed.norm
        self.layers = swin2d.layers
        self.norm = swin2d.norm
        self.swin_dim = swin2d.num_features
        del swin2d

        w = R3D_18_Weights.DEFAULT if pretrained else None
        base = r3d_18(weights=w)
        oc = base.stem[0]
        conv1 = nn.Conv3d(1, 64, kernel_size=oc.kernel_size, stride=oc.stride, padding=oc.padding, bias=False)
        with torch.no_grad():
            conv1.weight.copy_(oc.weight.mean(dim=1, keepdim=True))

        self.cnn_branch = nn.Sequential(
            conv1,
            base.stem[1],
            base.stem[2],
            base.layer1,
            base.layer2,
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
        )
        self.cnn_dim = 128
        del base

        self.head = nn.Linear(self.swin_dim + self.cnn_dim, num_classes)
        self.num_features = self.swin_dim + self.cnn_dim

    def forward(self, x):
        z_cnn = self.cnn_branch(x)
        t = self.patch_embed_3d(x)
        if self.dw_mix is not None:
            t = t + self.dw_mix(t)
        b, c, d, h, w = t.shape
        t = t.view(b, c, 4, 4, h, w)
        t = t.permute(0, 1, 2, 4, 3, 5)
        t = t.reshape(b, c, 4 * h, 4 * w)
        t = t.permute(0, 2, 3, 1)
        t = self.patch_embed_norm(t)
        for layer in self.layers:
            t = grad_checkpoint(layer, t, use_reentrant=False)
        t = self.norm(t)
        z_swin = t.mean(dim=[1, 2])
        z = torch.cat([z_swin, z_cnn], dim=1)
        return self.head(z)


class R3D18Expert(nn.Module):
    """R3D-18 para 3D binario (usado por Exp5 Pancreas; y plan B para Exp4 si hiciera falta)."""

    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        base = r3d_18(weights=weights)
        old_conv = base.stem[0]
        stem_conv = nn.Conv3d(
            1,
            64,
            kernel_size=(3, 7, 7),
            stride=(1, 2, 2),
            padding=(1, 3, 3),
            bias=False,
        )
        with torch.no_grad():
            stem_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        base.stem[0] = stem_conv
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        # Gradient checkpointing obligatorio para expertos 3D (consigna §8.1)
        for block in self.backbone:
            x = grad_checkpoint(block, x, use_reentrant=False)
        return self.head(x.flatten(1))


# ============================================================================
# Factory y carga de pesos
# ============================================================================
import glob

EXPERT_SPECS = {
    1: {"name": "exp1_nih", "num_classes": 5, "arch": "swin_tiny_nih"},
    2: {"name": "exp2_isic", "num_classes": 9, "arch": "efficientnet_b3"},
    3: {"name": "exp3_osteo", "num_classes": 5, "arch": "vgg16_bn"},
    4: {"name": "exp4_luna16", "num_classes": 2, "arch": "dcswinb3d"},
    5: {"name": "exp5_pancreas", "num_classes": 2, "arch": "r3d18"},
}


def build_expert(expert_id: int, pretrained_backbone: bool = True):
    spec = EXPERT_SPECS[int(expert_id)]
    arch = spec["arch"]
    num_classes = spec["num_classes"]

    if arch == "swin_tiny_nih":
        return SwinNIHClassifier(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "efficientnet_b3":
        return timm.create_model("efficientnet_b3", pretrained=pretrained_backbone, num_classes=num_classes)
    if arch == "vgg16_bn":
        return build_vgg16_bn(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "dcswinb3d":
        # En entrenamiento original: pretrained=True para timm/torchvision.
        # Al cargar checkpoint, pretrained_backbone puede ser False (los pesos vienen del ckpt).
        return DCSwinBStyle3D(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "r3d18":
        return R3D18Expert(num_classes=num_classes, pretrained=pretrained_backbone)
    raise ValueError(f"Arquitectura no soportada: {arch}")


def _default_checkpoint_candidates(weights_dir: str) -> Dict[int, List[str]]:
    """
    Candidatos por experto para tolerar nombres antiguos/nuevos de checkpoints.
    """
    return {
        1: [
            os.path.join(weights_dir, "Experts_2D", "Swin_NIH_5cls.pth"),
            os.path.join(weights_dir, "exp1_NIH_Swin_Tiny_best.pth"),
            os.path.join(weights_dir, "MaxViT_NIH_5cls.pth"),
            os.path.join(weights_dir, "exp1_NIH_SwinTiny_best.pth"),
            os.path.join(weights_dir, "exp1_NIH_LungMaxViT_best.pth"),
        ],
        2: [
            os.path.join(weights_dir, "exp2_ISIC_EfficientNetB3_best.pth"),
        ],
        3: [
            os.path.join(weights_dir, "exp3_Osteo_VGG16BN_best.pth"),
        ],
        4: [
            # Ruta real del Experto 4 en LUNA16_Swin3D_Training.ipynb
            os.path.join(weights_dir, "Experts_2D", "Expert4_LUNA16", "checkpoints", "expert4_*_best.pth"),
            # Legacy opcional (si alguna vez exportaste así)
            os.path.join(weights_dir, "exp4_LUNA16_3D_best.pth"),
        ],
        5: [
            os.path.join(weights_dir, "exp5_Pancreas_3D_best.pth"),
            os.path.join(weights_dir, "r3d18_pancreas_best_V2.pth"),
        ],
    }


def resolve_checkpoint(candidates: List[str]) -> str:
    # Soporta paths directos y patrones glob (e.g. expert4_*_best.pth)
    for p in candidates:
        if any(ch in p for ch in ['*', '?', '[']):
            matches = sorted(glob.glob(p))
            if matches:
                return matches[0]
        if os.path.exists(p):
            return p
    return ""


def load_weights(model: nn.Module, ckpt_path: str, map_location: str = "cpu", strict: bool = False):
    if not ckpt_path or not os.path.exists(ckpt_path):
        return False, "checkpoint no encontrado"
    raw = torch.load(ckpt_path, map_location=map_location)

    # Checkpoints de entrenamiento suelen ser dict con 'state_dict'
    if isinstance(raw, dict):
        if "state_dict" in raw:
            state = raw["state_dict"]
        elif "model_state_dict" in raw:
            state = raw["model_state_dict"]
        elif "model" in raw and isinstance(raw["model"], dict):
            state = raw["model"]
        else:
            state = raw
    else:
        state = raw

    try:
        model.load_state_dict(state, strict=strict)
        return True, "ok"
    except Exception as e:
        return False, f"load_state_dict fallo: {e}"

def freeze_and_eval(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model


def load_all_experts_from_drive(
    weights_dir: str,
    device: str = "cpu",
    strict: bool = False,
    pretrained_backbone: bool = False,
) -> Tuple[Dict[int, nn.Module], Dict[int, dict]]:
    """
    Crea y carga los 5 expertos con pesos desde Drive.

    Retorna:
      experts: dict[int, nn.Module]
      info: dict[int, {arch, ckpt, loaded, params}]
    """
    candidates = _default_checkpoint_candidates(weights_dir)
    experts = {}
    info = {}

    for eid in sorted(EXPERT_SPECS.keys()):
        model = build_expert(eid, pretrained_backbone=pretrained_backbone)
        ckpt_path = resolve_checkpoint(candidates[eid])
        loaded, msg = load_weights(model, ckpt_path, map_location="cpu", strict=strict)

        # Si Exp4 fue entrenado con plan B (R3D18) pero spec pide DCSwin, intentar fallback.
        if not loaded and int(eid) == 4:
            fallback = R3D18Expert(num_classes=2, pretrained=pretrained_backbone)
            loaded2, msg2 = load_weights(fallback, ckpt_path, map_location="cpu", strict=strict)
            if loaded2:
                model = fallback
                loaded, msg = loaded2, f"fallback R3D18 OK: {msg2}"

        model = freeze_and_eval(model).to(device)
        experts[eid] = model
        info[eid] = {
            "name": EXPERT_SPECS[eid]["name"],
            "arch": EXPERT_SPECS[eid]["arch"],
            "ckpt": ckpt_path,
            "loaded": loaded,
            "message": msg,
            "params": int(sum(p.numel() for p in model.parameters())),
        }

    return experts, info


def print_expert_load_report(info: Dict[int, dict]):
    print("=== Expert Load Report ===")
    for eid in sorted(info.keys()):
        row = info[eid]
        status = "OK" if row["loaded"] else "MISSING"
        print(
            f"Exp{eid} | {row['name']} | arch={row['arch']} | {status} | "
            f"params={row['params']:,} | ckpt={row['ckpt'] or 'N/A'}"
        )
print("Arquitecturas de expertos: definidas en el notebook.")


Arquitecturas de expertos: definidas en el notebook.


## Arquitecturas de expertos (embebidas)

La celda de código **anterior** define en el propio notebook las clases (**SwinNIHClassifier** / Swin-Tiny para NIH, EfficientNet-B3, VGG16-BN, **DCSwinBStyle3D para Exp4 LUNA16**, y R3D-18 para Exp5) y las funciones `load_all_experts_from_drive` / `print_expert_load_report`.


## Fase 1: Extracción de datos a disco local


In [4]:
def extract_datasets_colab(raw_dir=RAW_DIR, local_dest=LOCAL_DEST):
    """Copia ZIPs de Drive a /content/datasets/ y los descomprime."""
    if not os.path.exists(raw_dir):
        print(f'Ruta {raw_dir} no existe. No se extrae nada.')
        return
    zip_files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith('.zip')])
    print(f'Encontrados {len(zip_files)} zips.')
    for zip_name in zip_files:
        print('=' * 60)
        print(f'Procesando: {zip_name}')
        drive_zip_path = os.path.join(raw_dir, zip_name)
        dataset_name = os.path.splitext(zip_name)[0]
        unzip_dir = os.path.join(local_dest, dataset_name)
        local_zip_path = os.path.join(local_dest, zip_name)
        if os.path.isdir(unzip_dir) and len(os.listdir(unzip_dir)) > 0:
            print(f' Ya existe: {unzip_dir} (omitido).')
            continue
        print(' 1. Copiando ZIP...')
        shutil.copy2(drive_zip_path, local_zip_path)
        os.makedirs(unzip_dir, exist_ok=True)
        print(f' 2. Descomprimiendo en {unzip_dir}...')
        subprocess.run(['unzip', '-q', local_zip_path, '-d', unzip_dir], check=True)
        print(' 3. Borrando ZIP local.')
        os.remove(local_zip_path)
        # ZIPs internos
        for iz in glob.glob(os.path.join(unzip_dir, '**', '*.zip'), recursive=True):
            print(f' -> ZIP interno: {iz}')
            subprocess.run(['unzip', '-q', iz, '-d', os.path.dirname(iz)], check=True)
            os.remove(iz)
    print('\nExtracción completa.')

# Ejecutar una sola vez por sesión de Colab
extract_datasets_colab()


Encontrados 5 zips.
Procesando: ISIC 2019.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/ISIC 2019...
 3. Borrando ZIP local.
Procesando: Knee Osteoarthritis Classification.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Knee Osteoarthritis Classification...
 3. Borrando ZIP local.
Procesando: Luna16 Lung Cancer Dataset.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Luna16 Lung Cancer Dataset...
 3. Borrando ZIP local.
Procesando: NIH Chest X ray 14.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/NIH Chest X ray 14...
 3. Borrando ZIP local.
Procesando: Pancreas Cancer.zip
 1. Copiando ZIP...
 2. Descomprimiendo en /content/datasets/Pancreas Cancer...
 3. Borrando ZIP local.
 -> ZIP interno: /content/datasets/Pancreas Cancer/batch_1.zip

Extracción completa.


## Fase 2: AdaptivePreprocessor (Punto A — Carga nativa)
Carga archivos PNG/JPG/MHD/NIfTI y los convierte en tensores nativos:
- 2D: `[3, 224, 224]`
- 3D: `[1, 64, 64, 64]`

**Sin metadatos.** Solo mira la forma del tensor (`ndim`) para saber si es 2D o 3D.


In [5]:

import SimpleITK as sitk

def extract_luna_roi_64(mhd_path, coords_world, size=(64, 64, 64), hu_min=-1000.0, hu_max=400.0):
    try:
        img = sitk.ReadImage(mhd_path)
        orig_spacing = np.array(img.GetSpacing())
        orig_size = np.array(img.GetSize())
        new_spacing = np.array([1.0, 1.0, 1.0])
        new_size = (orig_size * orig_spacing / new_spacing).astype(int).tolist()

        resampler = sitk.ResampleImageFilter()
        resampler.SetOutputSpacing(new_spacing.tolist())
        resampler.SetSize(new_size)
        resampler.SetInterpolator(sitk.sitkLinear)
        resampler.SetOutputOrigin(img.GetOrigin())
        resampler.SetOutputDirection(img.GetDirection())
        img = resampler.Execute(img)

        vol = sitk.GetArrayFromImage(img).astype(np.float32)
        vol = np.clip(vol, hu_min, hu_max)
        vol = (vol - hu_min) / (hu_max - hu_min + 1e-8)

        voxel_coord = img.TransformPhysicalPointToIndex(coords_world)
        cz, cy, cx = voxel_coord[2], voxel_coord[1], voxel_coord[0]

        half = size[0] // 2
        d, h, w = vol.shape
        z0 = max(0, min(d - size[0], cz - half)); z1 = z0 + size[0]
        y0 = max(0, min(h - size[1], cy - half)); y1 = y0 + size[1]
        x0 = max(0, min(w - size[2], cx - half)); x1 = x0 + size[2]

        roi = vol[max(0,z0):z1, max(0,y0):y1, max(0,x0):x1]
        if roi.shape != size:
            p_d, p_h, p_w = max(0, size[0]-roi.shape[0]), max(0, size[1]-roi.shape[1]), max(0, size[2]-roi.shape[2])
            roi = np.pad(roi, ((0, p_d), (0, p_h), (0, p_w)), mode='constant')
        return roi[:size[0], :size[1], :size[2]]
    except Exception: return None

class AdaptivePreprocessor:
    def __init__(self, size_2d=(224, 224), size_3d=(64, 64, 64), hu_window_ct=(-1000, 400)):
        self.size_2d, self.size_3d = size_2d, size_3d
        self.hu_min, self.hu_max = hu_window_ct

    def __call__(self, input_val):
        if isinstance(input_val, dict):
            mhd, coords = input_val['mhd_path'], [input_val['x'], input_val['y'], input_val['z']]
            roi = extract_luna_roi_64(mhd, coords, size=self.size_3d, hu_min=self.hu_min, hu_max=self.hu_max)
            if roi is not None: return torch.from_numpy(roi).unsqueeze(0)
            return self._process_3d(mhd)
        
        path = str(input_val)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')): return self._process_2d(path)
        if path.lower().endswith('.mha'): return self._process_mha(path)
        return self._process_3d(path)

    def _process_2d(self, path):
        img = Image.open(path).convert('RGB')
        return T.Compose([T.Resize(self.size_2d), T.ToTensor()])(img)

    def _process_mha(self, path):
        itk_img = sitk.ReadImage(path)
        arr = sitk.GetArrayFromImage(itk_img).astype(np.float32)
        if len(itk_img.GetSize()) == 3 and itk_img.GetSize()[2] <= 1: arr = np.squeeze(arr)
        if arr.ndim == 2: return self._array_2d_to_tensor(arr)
        return self._volume_array_to_tensor(arr, path)

    def _array_2d_to_tensor(self, arr):
        if arr.max() > 1.5: arr = arr / 255.0 if arr.max() > 2.0 else (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        t = torch.from_numpy(arr).unsqueeze(0)
        if t.shape[1:] != torch.Size(self.size_2d): t = F.interpolate(t.unsqueeze(0), size=self.size_2d, mode='bilinear').squeeze(0)
        return t.repeat(3, 1, 1) if t.shape[0] == 1 else t[:3]

    def _volume_array_to_tensor(self, img_arr, path_hint=''):
        pre_norm = (float(np.nanmax(img_arr)) <= 1.5 and float(np.nanmin(img_arr)) >= -1e-2) or 'Pancreas' in str(path_hint).replace('\\', '/')
        if not pre_norm:
            img_arr = np.clip(img_arr, self.hu_min, self.hu_max)
            img_arr = (img_arr - self.hu_min) / (self.hu_max - self.hu_min + 1e-8)
        else: img_arr = np.clip(img_arr, 0.0, 1.0)
        if img_arr.shape == self.size_3d: return torch.from_numpy(img_arr).unsqueeze(0)
        t = torch.tensor(img_arr).unsqueeze(0).unsqueeze(0)
        return F.interpolate(t, size=self.size_3d, mode='trilinear').squeeze(0)

    def _process_3d(self, path):
        if path.lower().endswith('.npz'):
            d = np.load(path); k = 'volume' if 'volume' in d else ('Z' if 'Z' in d else list(d.keys())[0])
            return self._volume_array_to_tensor(d[k].astype(np.float32), path_hint=path)
        img_arr = sitk.GetArrayFromImage(sitk.ReadImage(path)).astype(np.float32) if path.lower().endswith('.mhd') else np.transpose(nib.load(path).get_fdata().astype(np.float32), (2,1,0))
        return self._volume_array_to_tensor(img_arr, path_hint=path)


AdaptivePreprocessor listo.


## Fase 3: Escáner de archivos
Función que recorre recursivamente un directorio y retorna los paths de los archivos médicos.


In [6]:
VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.mha', '.mhd', '.nii', '.nii.gz')

def scan_dataset_files(root_path, max_files=None):
    """Escanea recursivamente y retorna paths de archivos médicos válidos."""
    files = []
    for dirpath, _, filenames in os.walk(root_path):
        for fname in sorted(filenames):
            if fname.lower().endswith(VALID_EXTENSIONS):
                files.append(os.path.join(dirpath, fname))
    if max_files:
        files = files[:max_files]
    return files

# Verificación rápida
for name, (path, did) in DATASET_ROOTS.items():
    if os.path.exists(path):
        count = len(scan_dataset_files(path, max_files=None))
        print(f'  {name} (ID={did}): {count} archivos en {path}')
    else:
        print(f'  {name} (ID={did}): RUTA NO ENCONTRADA — {path}')


  NIH (ID=0): 112120 archivos en /content/datasets/NIH Chest X ray 14
  ISIC (ID=1): 25331 archivos en /content/datasets/ISIC 2019
  Osteo (ID=2): 12712 archivos en /content/datasets/Knee Osteoarthritis Classification
  LUNA16 (ID=3): 889 archivos en /content/datasets/Luna16 Lung Cancer Dataset
  Pancreas (ID=4): 557 archivos en /content/datasets/Pancreas Cancer


## Opcional — Cache en Drive: tensores preprocesados como `.npz`

Convierte cada archivo médico (mismo pipeline que `AdaptivePreprocessor`) y guarda el array **`float32`** en Google Drive bajo `PROC_BASE` (o la ruta que definas).

- Cada `.npz` contiene: `x` (tensor NCHW o CDHW), `dataset_id`, `source` (ruta absoluta original).
- Usa `MAX_NPZ_PER_DATASET` para no llenar el disco en una sola pasada; pon `None` para todo el dataset.
- Si el archivo ya existe y `SKIP_EXISTING_NPZ=True`, se omite.

In [ ]:
# --- Guardar imágenes/volúmenes ya preprocesados como .npz en Drive ---
import hashlib
import re

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kw: x

# Raíz en Drive (misma base que PROC_BASE del inicio del notebook)
NPZ_CACHE_ROOT = os.path.join(PROC_BASE, "npz_adaptive_preprocessed")
SAVE_NPZ_TO_DRIVE = False  # pon True para ejecutar la exportación
MAX_NPZ_PER_DATASET = 50   # None = todos los archivos encontrados
SKIP_EXISTING_NPZ = True


def _npz_dest_path(cache_root: str, ds_name: str, src_path: str) -> str:
    h = hashlib.md5(os.path.abspath(src_path).encode("utf-8")).hexdigest()[:12]
    base = re.sub(r"[^\w\-.]+", "_", os.path.basename(src_path))
    base = (base[:100] if base else "file") + f"_{h}.npz"
    return os.path.join(cache_root, ds_name, base)


def export_preprocessed_to_npz_on_drive(
    roots: dict,
    preprocessor,
    out_root: str = NPZ_CACHE_ROOT,
    max_per_dataset: int | None = MAX_NPZ_PER_DATASET,
    skip_existing: bool = SKIP_EXISTING_NPZ,
):
    """
    Recorre cada dataset en `roots`, aplica `preprocessor(path)` y guarda `.npz` en `out_root/<nombre_dataset>/`.
    """
    os.makedirs(out_root, exist_ok=True)
    total_saved = 0
    for ds_name, (root_path, did) in roots.items():
        if not os.path.isdir(root_path):
            print(f"[omitido] {ds_name}: no existe {root_path}")
            continue
        files = scan_dataset_files(root_path)
        if max_per_dataset is not None:
            files = files[: int(max_per_dataset)]
        ds_out = os.path.join(out_root, ds_name)
        os.makedirs(ds_out, exist_ok=True)
        print(f"[{ds_name}] exportando {len(files)} archivos → {ds_out}")
        for fpath in tqdm(files, desc=f"npz {ds_name}"):
            out_npz = _npz_dest_path(out_root, ds_name, fpath)
            if skip_existing and os.path.isfile(out_npz):
                continue
            try:
                t = preprocessor(fpath)
                arr = t.detach().cpu().numpy().astype(np.float32)
                np.savez_compressed(
                    out_npz,
                    x=arr,
                    dataset_id=np.int32(did),
                    source=np.array(os.path.abspath(fpath)),
                )
                total_saved += 1
            except Exception as ex:
                print(f"  ERROR {fpath}: {ex}")
    print(f"Listo. Archivos .npz guardados (nuevos): {total_saved} | raíz: {out_root}")
    return out_root


if SAVE_NPZ_TO_DRIVE:
    export_preprocessed_to_npz_on_drive(
        DATASET_ROOTS,
        preprocessor,
        out_root=NPZ_CACHE_ROOT,
        max_per_dataset=MAX_NPZ_PER_DATASET,
        skip_existing=SKIP_EXISTING_NPZ,
    )
else:
    print(
        "Cache .npz en Drive desactivada. Pon SAVE_NPZ_TO_DRIVE=True y vuelve a ejecutar. "
        f"Destino previsto: {NPZ_CACHE_ROOT}"
    )


## Fase 4: DataLoader Mixto (Puntos B y C)

### B: `MixedMedicalDataset`
Dataset unificado que mezcla los 5 datasets. Retorna `(tensor, dataset_id)`.

### C: `mixed_collate_fn`
Collate personalizado que **NO** intenta apilar tensores de distinta forma.
Devuelve listas crudas para que el modelo procese cada muestra individualmente.


In [7]:
import pandas as pd

# -------- Etiquetas por dataset (misma lógica de notebooks de expertos) --------
NIH_CLASSES_14 = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule',
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
ISIC_CLASSES_8 = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']


def _safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None


def build_dataset_label_index(roots):
    """
    Construye índices de etiquetas por dataset para usar en el loader mixto.
    - NIH: multi-label (14) desde Data_Entry* (Finding Labels).
    - ISIC: single-label (8) desde CSV one-hot (MEL..SCC), excluye UNK.
    - Osteo: clase KL desde carpeta padre (0..4).
    - LUNA16: label binario por seriesuid desde candidates*.csv (class 0/1), agregado por serie.
    - Pancreas: label en NPZ (si aplica) o inferencia por nombre/ruta.
    """
    idx = {'nih': {}, 'isic': {}, 'osteo': {}, 'luna16': {}, 'meta': {}}

    # NIH
    nih_root = roots.get('NIH', (None, None))[0]
    if nih_root and os.path.isdir(nih_root):
        nih_csv = None
        for c in glob.glob(os.path.join(nih_root, '**', '*.csv'), recursive=True):
            bn = os.path.basename(c).lower()
            if 'data_entry' in bn and '2017' in bn:
                nih_csv = c
                break
        if nih_csv:
            df = _safe_read_csv(nih_csv)
            if df is not None and 'Image Index' in df.columns and 'Finding Labels' in df.columns:
                for _, r in df.iterrows():
                    img = str(r['Image Index'])
                    labels_txt = str(r['Finding Labels'])
                    vec = np.zeros(len(NIH_CLASSES_14), dtype=np.float32)
                    toks = [t.strip() for t in labels_txt.split('|') if t.strip()]
                    for i, cls in enumerate(NIH_CLASSES_14):
                        vec[i] = 1.0 if cls in toks else 0.0
                    idx['nih'][img] = vec
                idx['meta']['nih_csv'] = nih_csv

    # ISIC
    isic_root = roots.get('ISIC', (None, None))[0]
    if isic_root and os.path.isdir(isic_root):
        isic_csv = None
        for c in glob.glob(os.path.join(isic_root, '**', '*.csv'), recursive=True):
            df = _safe_read_csv(c)
            if df is not None and all(col in df.columns for col in ISIC_CLASSES_8):
                isic_csv = c
                break
        if isic_csv:
            df = _safe_read_csv(isic_csv)
            if df is not None:
                if 'UNK' in df.columns:
                    df = df[df['UNK'] != 1.0].reset_index(drop=True)
                img_col = df.columns[0]
                avail = [c for c in ISIC_CLASSES_8 if c in df.columns]
                for _, r in df.iterrows():
                    name = str(r[img_col])
                    row = r[avail].to_numpy(dtype=np.float32)
                    if len(row) == len(ISIC_CLASSES_8):
                        lbl = int(np.argmax(row))
                        idx['isic'][name] = lbl
                idx['meta']['isic_csv'] = isic_csv

    # Osteoarthritis: indice explicito identico al notebook de entrenamiento
    # Estructura: root/KLGrade/KLGrade/0..4/imagen.jpg (o .mha tras conversion)
    osteo_root = roots.get('Osteo', (None, None))[0]
    if osteo_root and os.path.isdir(osteo_root):
        # Intenta la ruta doble KLGrade/KLGrade (estructura original del zip)
        klgrade_dir = os.path.join(osteo_root, 'KLGrade', 'KLGrade')
        if not os.path.isdir(klgrade_dir):
            # Fallback: puede estar ya en la raiz o un nivel
            klgrade_dir = os.path.join(osteo_root, 'KLGrade')
        if not os.path.isdir(klgrade_dir):
            klgrade_dir = osteo_root
        for grade_folder in sorted(os.listdir(klgrade_dir)):
            gp = os.path.join(klgrade_dir, grade_folder)
            if not os.path.isdir(gp):
                continue
            try:
                grade = int(grade_folder)
            except ValueError:
                continue
            if grade not in [0, 1, 2, 3, 4]:
                continue
            for fname in os.listdir(gp):
                fpath = os.path.normpath(os.path.join(gp, fname))
                idx['osteo'][fpath] = grade
        idx['meta']['osteo_dir'] = klgrade_dir

    # LUNA16 (por serie)
    luna_root = roots.get('LUNA16', (None, None))[0]
    if luna_root and os.path.isdir(luna_root):
        cand = None
        all_csv = glob.glob(os.path.join(luna_root, '**', '*.csv'), recursive=True)
        cand = next((c for c in all_csv if 'candidates_v2' in os.path.basename(c).lower()), None)
        if cand is None:
            cand = next((c for c in all_csv if 'candidates' in os.path.basename(c).lower()), None)
        if cand:
            df = _safe_read_csv(cand)
            if df is not None:
                cols_map = {c.lower(): c for c in df.columns}
                if 'seriesuid' in cols_map and 'class' in cols_map:
                    s_col = cols_map['seriesuid']
                    c_col = cols_map['class']
                    df[c_col] = pd.to_numeric(df[c_col], errors='coerce').fillna(0).astype(int)
                    # Agregado por serie para mapear volumen completo -> etiqueta binaria
                    agg = df.groupby(s_col)[c_col].max().to_dict()
                    mhd_map = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(luna_root, '**', '*.mhd'), recursive=True)}
                    idx['luna16'] = [{'mhd_path': mhd_map[s], 'x': float(r['coordX']), 'y': float(r['coordY']), 'z': float(r['coordZ']), 'label': int(r['class'])} for _, r in df.iterrows() if str(r['seriesuid']) in mhd_map]
                    idx['meta']['luna_csv'] = cand

    return idx


def _infer_osteo_label_from_path(path):
    parts = os.path.normpath(path).split(os.sep)
    for p in reversed(parts):
        if p.isdigit() and int(p) in [0, 1, 2, 3, 4]:
            return int(p)
        up = p.upper()
        if up.startswith('KL') and up[2:].isdigit():
            v = int(up[2:])
            if v in [0, 1, 2, 3, 4]:
                return v
    return None


def _infer_pancreas_label_from_path_or_npz(path):
    low = path.lower()
    if low.endswith('.npz'):
        try:
            d = np.load(path)
            if 'label' in d:
                return int(d['label'])
            if 'y' in d:
                return int(d['y'])
        except Exception:
            pass
    # fallback por nombre/carpeta
    pos_tokens = ['cancer', 'tumor', 'positive', 'pdac', 'malig']
    neg_tokens = ['normal', 'negative', 'benign', 'non_tumor']
    if any(t in low for t in pos_tokens):
        return 1
    if any(t in low for t in neg_tokens):
        return 0
    return None


def resolve_task_label(path, dataset_id, label_index):
    """Devuelve etiqueta de tarea por muestra según dataset experto."""
    p = os.path.normpath(path)
    fname = os.path.basename(p)
    stem = os.path.splitext(fname)[0]

    if dataset_id == 0:  # NIH multilabel (14)
        vec = label_index['nih'].get(fname)
        if vec is None:
            # fallback por stem por si hay extensión distinta
            for k, v in label_index['nih'].items():
                if os.path.splitext(k)[0] == stem:
                    vec = v
                    break
        return vec

    if dataset_id == 1:  # ISIC 8 clases
        lbl = label_index['isic'].get(stem)
        if lbl is None:
            # fallback si viene con .jpg en el CSV
            lbl = label_index['isic'].get(fname)
        return lbl

    if dataset_id == 2:  # Osteo KL — indice explicito primero, heuristica como fallback
        lbl = label_index.get('osteo', {}).get(p)
        if lbl is None:
            lbl = _infer_osteo_label_from_path(p)
        return lbl

    if dataset_id == 3:  # LUNA16 binario por seriesuid
        seriesuid = stem
        return label_index['luna16'].get(seriesuid)

    if dataset_id == 4:  # Pancreas binario
        return _infer_pancreas_label_from_path_or_npz(p)

    return None


class MixedMedicalDataset(torch.utils.data.Dataset):
    """Dataset unificado: mezcla 2D/3D y opcionalmente devuelve etiqueta de tarea por dataset."""
    def __init__(self, file_list, dataset_ids, preprocessor, label_index=None, include_task_label=False):
        self.file_list = file_list
        self.dataset_ids = dataset_ids
        self.preprocessor = preprocessor
        self.label_index = label_index
        self.include_task_label = include_task_label

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fpath = self.file_list[idx]
        did = int(self.dataset_ids[idx])
        tensor = self.preprocessor(fpath)
        if not self.include_task_label:
            return tensor, did

        tlabel = resolve_task_label(fpath, did, self.label_index if self.label_index is not None else {})
        return tensor, did, tlabel, fpath


def mixed_collate_fn(batch):
    """
    Collate personalizado (Punto C).
    - modo clásico: retorna (tensors, dataset_ids)
    - modo extendido: retorna (tensors, dataset_ids, task_labels, paths)
    """
    tensors = [b[0] for b in batch]
    dataset_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)

    if len(batch[0]) == 2:
        return tensors, dataset_ids

    task_labels = [b[2] for b in batch]
    paths = [b[3] for b in batch]
    return tensors, dataset_ids, task_labels, paths


def build_router_dataloader(roots, preprocessor, batch_size=8, num_workers=2, sample_cap=1000, include_task_label=False):
    """
    Construye DataLoader mixto balanceado.
    Si include_task_label=True, agrega etiquetas de tarea por dataset (no dummy).
    """
    all_files, all_ids = [], []
    for name, (path, did) in roots.items():
        files = scan_dataset_files(path)
        if len(files) > sample_cap:
            random.seed(42)
            files = random.sample(files, sample_cap)
            print(f'  [{name}] Balanceado: reducido a {sample_cap} muestras (ID={did})')
        else:
            print(f'  [{name}] {len(files)} archivos cargados en total (ID={did})')
        all_files.extend(files)
        all_ids.extend([did] * len(files))

    label_index = build_dataset_label_index(roots) if include_task_label else None
    if include_task_label:
        print('Indices de etiquetas construidos:')
        print('  NIH:', len(label_index.get('nih', {})))
        print('  ISIC:', len(label_index.get('isic', {})))
        print('  Osteo:', len(label_index.get('osteo', {})), 'imagenes indexadas')
        print('  LUNA16 series:', len(label_index.get('luna16', {})))

    dataset = MixedMedicalDataset(
        all_files,
        all_ids,
        preprocessor,
        label_index=label_index,
        include_task_label=include_task_label,
    )
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=mixed_collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )
    print(f'\nDataLoader listo: {len(dataset)} muestras, batch_size={batch_size}, include_task_label={include_task_label}')
    return loader


print('MixedMedicalDataset + labels por dataset definidos.')


MixedMedicalDataset + labels por dataset definidos.


In [8]:
# ============================================================
# VALIDACION: etiquetas por dataset correctamente asociadas
# ============================================================
# Ejecutar SOLO en Colab, con los 5 datasets disponibles en LOCAL_DEST.
# Imprime para cada dataset:
#   - cuantas etiquetas cargaron desde CSV/carpeta/npz
#   - N muestras verificadas con etiqueta valida vs None
#   - una muestra ejemplo por dataset

import os, glob
import numpy as np

def validate_dataset_labels(roots, n_check=10, seed=42):
    print("=" * 70)
    print("VALIDACION DE ETIQUETAS POR DATASET")
    print("=" * 70)

    rng = np.random.default_rng(seed)
    label_index = build_dataset_label_index(roots)

    for name, (root, did) in roots.items():
        print(f"\n[{name}] dataset_id={did} | raiz={root}")

        if not os.path.isdir(root):
            print("  NO DISPONIBLE (directorio no existe)")
            continue

        files = scan_dataset_files(root)
        if not files:
            print("  Sin archivos encontrados.")
            continue

        sample = files if len(files) <= n_check else rng.choice(files, size=n_check, replace=False).tolist()

        found, missing, examples = 0, 0, []
        for f in sample:
            lbl = resolve_task_label(f, did, label_index)
            if lbl is not None:
                found += 1
                if len(examples) < 3:
                    fname = os.path.basename(f)
                    if isinstance(lbl, np.ndarray):
                        lbl_str = f"multilabel len={len(lbl)} nonzero={int(np.sum(lbl))}"
                    else:
                        lbl_str = str(lbl)
                    examples.append(f"  {fname} -> {lbl_str}")
            else:
                missing += 1
                if missing <= 2:
                    examples.append(f"  {os.path.basename(f)} -> NONE")

        total = len(sample)
        pct = 100.0 * found / total if total else 0.0
        print(f"  Archivos chequeados: {total}")
        print(f"  Con etiqueta: {found}/{total} ({pct:.1f}%)")
        print(f"  Sin etiqueta: {missing}/{total}")

        if did == 0:
            n_idx = len(label_index.get('nih', {}))
            print(f"  Entradas en indice NIH (CSV): {n_idx}")
            if 'nih_csv' in label_index.get('meta', {}):
                print(f"  CSV usado: {label_index['meta']['nih_csv']}")
        elif did == 1:
            n_idx = len(label_index.get('isic', {}))
            print(f"  Entradas en indice ISIC (CSV): {n_idx}")
            if 'isic_csv' in label_index.get('meta', {}):
                print(f"  CSV usado: {label_index['meta']['isic_csv']}")
        elif did == 2:
            n_idx = len(label_index.get('osteo', {}))
            print(f"  Imagenes indexadas en indice Osteo: {n_idx}")
            osteo_dir = label_index.get('meta', {}).get('osteo_dir', 'N/A')
            print(f"  Directorio KLGrade usado: {osteo_dir}")
        elif did == 3:
            n_idx = len(label_index.get('luna16', {}))
            print(f"  Series con etiqueta LUNA16 (candidates CSV): {n_idx}")
            if 'luna_csv' in label_index.get('meta', {}):
                print(f"  CSV usado: {label_index['meta']['luna_csv']}")
        elif did == 4:
            print("  Etiqueta desde NPZ (key 'label'/'y') o token de nombre/ruta.")

        print("  Ejemplos:")
        for ex in examples:
            print(ex)

        # Alerta si mas del 20% no tiene etiqueta
        if pct < 80.0 and total >= 5:
            print(f"  ADVERTENCIA: solo {pct:.1f}% de muestras tienen etiqueta.")
            if did == 0:
                print("  -> Verifica que Data_Entry_2017.csv este en la raiz de NIH.")
            elif did == 1:
                print("  -> Verifica que el CSV de ISIC con columnas MEL..SCC este disponible.")
            elif did == 2:
                print("  -> Verifica que la estructura KLGrade/KLGrade/0..4/ exista.")
            elif did == 3:
                print("  -> Verifica que candidates_V2.csv este en la raiz de LUNA16.")
            elif did == 4:
                print("  -> Verifica que los NPZ tengan clave 'label' o que la ruta incluya tokens positivo/negativo.")

    print("\n" + "=" * 70)
    print("FIN VALIDACION")
    print("=" * 70)


# Ejecutar:
validate_dataset_labels(DATASET_ROOTS, n_check=15)

VALIDACION DE ETIQUETAS POR DATASET

[NIH] dataset_id=0 | raiz=/content/datasets/NIH Chest X ray 14
  Archivos chequeados: 15
  Con etiqueta: 15/15 (100.0%)
  Sin etiqueta: 0/15
  Entradas en indice NIH (CSV): 112120
  CSV usado: /content/datasets/NIH Chest X ray 14/Data_Entry_2017.csv
  Ejemplos:
  00001338_003.png -> multilabel len=14 nonzero=1
  00001490_000.png -> multilabel len=14 nonzero=0
  00008150_000.png -> multilabel len=14 nonzero=0

[ISIC] dataset_id=1 | raiz=/content/datasets/ISIC 2019
  Archivos chequeados: 15
  Con etiqueta: 15/15 (100.0%)
  Sin etiqueta: 0/15
  Entradas en indice ISIC (CSV): 25331
  CSV usado: /content/datasets/ISIC 2019/ISIC_2019_Training_GroundTruth.csv
  Ejemplos:
  ISIC_0055285.jpg -> 1
  ISIC_0014624_downsampled.jpg -> 4
  ISIC_0028412.jpg -> 0

[Osteo] dataset_id=2 | raiz=/content/datasets/Knee Osteoarthritis Classification
  Archivos chequeados: 15
  Con etiqueta: 7/15 (46.7%)
  Sin etiqueta: 8/15
  Etiqueta inferida desde carpeta KLGrade (0-4).

## Fase 5: SwitchablePatchEmbed (Puntos D → I)

Este módulo implementa el **corrimiento de ventana** que el profesor indicó:
- **D**: Foto 2D → parches de 16×16
- **E**: Volumen 3D → cubitos de 8×8×8
- **F, G**: Proyección lineal → cada parche se convierte en un vector de **192 dimensiones**
- **H**: Padding de secuencia (todos los batches al mismo largo)
- **I**: Se agrega Token CLS + Positional Embeddings

**Resultado:** Tensor homogéneo `[B, 513, 192]` listo para el Transformer.


In [9]:
from monai.networks.blocks import PatchEmbed
import torch
import torch.nn as nn

class SwitchablePatchEmbed(nn.Module):
    """
    Switchable Patch Embedding (SPE) — Pasos D→I.
    Versión corregida y más robusta.
    """
    def __init__(self, embed_dim=192, patch_size_2d=16, patch_size_3d=8, in_channels_2d=3):
        super().__init__()
        self.embed_dim = embed_dim

        # D: Patch Embedding 2D
        self.patch_embed_2d = PatchEmbed(
            spatial_dims=2,
            in_chans=in_channels_2d,      # ← Correcto: in_chans
            patch_size=patch_size_2d,
            embed_dim=embed_dim           # ← Correcto: embed_dim
        )

        # E: Patch Embedding 3D
        self.patch_embed_3d = PatchEmbed(
            spatial_dims=3,
            in_chans=1,
            patch_size=patch_size_3d,
            embed_dim=embed_dim
        )

        # 1 (CLS) + max(14*14, 8**3) = 1 + 512 = 513 tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 513, embed_dim))

    def _patch_tokens_to_sequence(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        """MONAI puede devolver [B,N,D] o tensores espaciales [B,H',W',D] / [B,D,H',W']."""
        x = patch_tokens
        if x.dim() == 3 and x.size(-1) == self.embed_dim:
            return x.squeeze(0) if x.size(0) == 1 else x.reshape(-1, self.embed_dim)
        if x.dim() >= 3 and x.size(-1) == self.embed_dim:
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 4 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 5 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        raise RuntimeError(f'Forma de patch tokens no soportada: {tuple(patch_tokens.shape)}')

    def forward(self, batch_tensors):
        batch_size = len(batch_tensors)
        tokens_list = []

        for sample in batch_tensors:
            sample = sample.unsqueeze(0)  # [1, C, ...]

            if sample.ndim == 4:
                if sample.shape[1] == 1:
                    sample = sample.repeat(1, 3, 1, 1)
                patch_tokens = self.patch_embed_2d(sample)
            elif sample.ndim == 5:
                patch_tokens = self.patch_embed_3d(sample)
            else:
                raise ValueError(f'Tensor de entrada inválido (ndim={sample.ndim}): {tuple(sample.shape)}')

            seq = self._patch_tokens_to_sequence(patch_tokens)
            tokens_list.append(seq)

        # H: Padding
        padded = torch.nn.utils.rnn.pad_sequence(tokens_list, batch_first=True)

        # Máscara (True = válido)
        lengths = torch.tensor([t.size(0) for t in tokens_list], device=padded.device)
        max_len = padded.size(1)
        mask = torch.arange(max_len, device=padded.device)[None, :] < lengths[:, None]

        # I: CLS + Positional
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        final_tokens = torch.cat((cls_tokens, padded), dim=1)

        cls_mask = torch.ones(batch_size, 1, dtype=torch.bool, device=mask.device)
        final_mask = torch.cat((cls_mask, mask), dim=1)

        final_tokens = final_tokens + self.pos_embed[:, :final_tokens.size(1), :]

        return final_tokens, final_mask

## Fase 6: VisionRouter Completo (Puntos J → M)

El modelo completo que orquesta todo:
- **J**: 12 bloques de Self-Attention (`TransformerEncoder`)
- **K**: Extracción del Token CLS final (posición 0)
- **L**: Vector de representación global `1×192`
- **M**: Capa lineal `192 → 5` para decidir a qué experto enviar

El modelo retorna `(logits, cls_vector)` para poder:
1. Entrenar el router con CrossEntropyLoss
2. Extraer los CLS para el ablation study


In [10]:
class VisionRouter(nn.Module):
    def __init__(self, embed_dim=192, num_experts=5, num_layers=12, pretrained=True):
        super().__init__()

        self.patch_embed = SwitchablePatchEmbed(embed_dim=embed_dim)

        # Usamos ViT-Tiny de timm (mucho más optimizado)
        self.vit = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=pretrained,
            num_classes=0,           # sin cabeza de clasificación
            global_pool='',          # no pooling
            img_size=224,            # solo referencia, no lo usamos realmente
        )

        # Reemplazamos el patch_embed original de timm por nuestro Switchable
        self.vit.patch_embed = nn.Identity()

        # Si queremos controlar el número de layers (opcional)
        if num_layers < 12:
            self.vit.blocks = self.vit.blocks[:num_layers]

        # Cabeza del router
        self.router_head = nn.Linear(embed_dim, num_experts)

    def forward(self, batch_tensors):
        # A → I: Patch + CLS + Positional
        x, mask = self.patch_embed(batch_tensors)   # [B, seq_len+1, 192]

        # timm ViT espera entrada sin CLS token + positional ya incluido
        # Como nosotros ya agregamos CLS y positional, pasamos directamente

        # Opción más limpia: forward manual solo de los bloques
        for blk in self.vit.blocks:
            x = blk(x)

        # K/L: Extraer CLS token (posición 0)
        cls_token = x[:, 0]                    # [B, 192]

        # M: Router head
        logits = self.router_head(cls_token)   # [B, 5]

        return logits, cls_token

print('VisionRouter definido (Pasos J→M).')


VisionRouter definido (Pasos J→M).


## Fase 7: Prueba con datos dummy (sin disco)
Verificamos que el pipeline completo funciona antes de usar datos reales.


In [11]:
# ---------- Prueba rápida ----------
model = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=False) # Pretrained False por velocidad en prueba
model.eval()

# Simulando un batch mixto (salida del mixed_collate_fn):
dummy_batch = [
    torch.randn(3, 224, 224),   # Imagen 2D (NIH, ISIC, Osteo)
    torch.randn(1, 64, 64, 64), # Volumen 3D (LUNA16, Pancreas)
    torch.randn(3, 224, 224),   # Imagen 2D
]

with torch.no_grad():
    logits, cls_features = model(dummy_batch)

print(f'Logits shape:       {logits.shape}')        # → [3, 5]
print(f'CLS features shape: {cls_features.shape}')  # → [3, 192]
print(f'Decisiones:         {logits.argmax(dim=1)}') # → tensor de 3 IDs de experto
print()
print('Pipeline A→M funciona correctamente.')
print(f'   Cada muestra produce un vector 1×192 y una decisión entre {logits.shape[1]} expertos.')

del model  # liberar VRAM


Logits shape:       torch.Size([3, 5])
CLS features shape: torch.Size([3, 192])
Decisiones:         tensor([1, 1, 1])

Pipeline A→M funciona correctamente.
   Cada muestra produce un vector 1×192 y una decisión entre 5 expertos.


## Fase 8: Extracción de CLS Tokens a Disco

**Recomendación del profesor (Fase 0 del Ablation Study):**

Una vez que el backbone ViT esté entrenado (o cargado de `timm`), pasamos
**todo el dataset** por el Router y guardamos los vectores `1×192` en disco.

Esto permite hacer el estudio de ablación (Linear vs GMM vs NB vs k-NN)
en segundos, sin re-procesar las imágenes pesadas.


In [12]:
def extract_and_save_cls_features(dataloader, model, save_path, device='cpu'):
    """
    Fase 0 del Ablation Study:
    Extrae los vectores CLS (1×192) de todo el dataset y los guarda en disco.
    """
    model.to(device)
    model.eval()
    all_features = []
    all_labels = []

    print(f'Extrayendo CLS features → {save_path}')
    with torch.no_grad():
        with autocast():  # ⚡ Acelera el Transformer enormemente en GPU
            for i, (tensors, ids) in enumerate(dataloader):
                # Mover tensores al dispositivo
                tensors = [t.to(device) for t in tensors]
                _, cls_features = model(tensors)
                all_features.append(cls_features.cpu())
                all_labels.append(ids.cpu())
                if (i + 1) % 10 == 0:
                    print(f'  Procesados {i + 1} batches ...')

    features_array = torch.cat(all_features, dim=0).numpy()
    labels_array = torch.cat(all_labels, dim=0).numpy()

    print(f'✅ Extracción Completa: {features_array.shape[0]} muestras × {features_array.shape[1]} dims')
    print(f'   Distribución Total: {dict(Counter(labels_array))}')

    # --- Split 80/20 y guardado en formato NumPy (.npy) como pidió el profesor ---
    print('\nDividiendo en Train (80%) y Val (20%) y guardando como .npy...')
    Z_train, Z_val, y_train, y_val = train_test_split(
        features_array, labels_array, test_size=0.20, random_state=42, stratify=labels_array
    )

    os.makedirs(save_path, exist_ok=True)

    np.save(os.path.join(save_path, 'Z_train.npy'), Z_train)
    np.save(os.path.join(save_path, 'Z_val.npy'), Z_val)
    np.save(os.path.join(save_path, 'y_train_expert.npy'), y_train)
    np.save(os.path.join(save_path, 'y_val_expert.npy'), y_val)

    print(f'✅ Guardado en: {save_path}')
    print(f'   Train: Z={Z_train.shape}, y={dict(Counter(y_train))}')
    print(f'   Val:   Z={Z_val.shape}, y={dict(Counter(y_val))}')

    return Z_train, Z_val, y_train, y_val

print('extract_and_save_cls_features definida con división Train/Val.')


extract_and_save_cls_features definida con división Train/Val.


## Fase 9: Ejecución con datos reales

Construimos el DataLoader con los 5 datasets y ejecutamos la extracción.

> **Nota:** Para pruebas rápidas, usa `max_per_dataset=10`.
> Para la extracción final, pon `max_per_dataset=None`.


In [13]:
# ---------- Construir DataLoader Mixto ----------
# Para prueba rápida: max_per_dataset=10
# Para extracción completa: max_per_dataset=None

loader = build_router_dataloader(
    roots=DATASET_ROOTS,
    preprocessor=preprocessor,
    batch_size=16,      # ⚡ Batch más grande gracias a las optimizaciones
    num_workers=2,
    sample_cap=1000     # ← Balanceo robusto
)


  [NIH] Balanceado: reducido a 1000 muestras (ID=0)
  [ISIC] Balanceado: reducido a 1000 muestras (ID=1)
  [Osteo] Balanceado: reducido a 1000 muestras (ID=2)
  [LUNA16] 889 archivos cargados en total (ID=3)
  [Pancreas] 557 archivos cargados en total (ID=4)

DataLoader listo: 4446 muestras, batch_size=16, include_task_label=False


In [14]:
# ---------- Instanciar modelo y extraer CLS ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

router = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=True)

SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')

Z_train, Z_val, y_train, y_val = extract_and_save_cls_features(
    dataloader=loader,
    model=router,
    save_path=SAVE_DIR,
    device=device
)


Dispositivo: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Extrayendo CLS features → /content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/ablation_data_v1


/tmp/ipykernel_6203/1756661291.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # ⚡ Acelera el Transformer enormemente en GPU
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Procesados 10 batches ...
  Procesados 20 batches ...
  Procesados 30 batches ...
  Procesados 40 batches ...
  Procesados 50 batches ...
  Procesados 60 batches ...
  Procesados 70 batches ...
  Procesados 80 batches ...
  Procesados 90 batches ...
  Procesados 100 batches ...
  Procesados 110 batches ...
  Procesados 120 batches ...
  Procesados 130 batches ...
  Procesados 140 batches ...
  Procesados 150 batches ...
  Procesados 160 batches ...
  Procesados 170 batches ...
  Procesados 180 batches ...
  Procesados 190 batches ...
  Procesados 200 batches ...
  Procesados 210 batches ...
  Procesados 220 batches ...
  Procesados 230 batches ...
  Procesados 240 batches ...
  Procesados 250 batches ...
  Procesados 260 batches ...
  Procesados 270 batches ...
✅ Extracción Completa: 4446 muestras × 192 dims
   Distribución Total: {np.int64(2): 1000, np.int64(3): 889, np.int64(1): 1000, np.int64(4): 557, np.int64(0): 1000}

Dividiendo en Train (80%) y Val (20%) y guardando como .npy.

## Fase 10: Estudio de Ablación (Routing Mechanisms)

Una vez guardados los CLS en disco, comparamos qué clasifica mejor el flujo:
1. **Linear + Softmax** (Capa densa, entrenada con gradiente)
2. **GMM** (Gaussian Mixture Model, sklearn)
3. **Naive Bayes** (GaussianNB, sklearn)
4. **k-NN** (FAISS, distancia coseno)

> La implementación detallada de cada mecanismo va en el notebook 04.


In [16]:
# --- Fase 19: entrenamiento del router con métricas completas ---

import math


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _switch_aux_loss(router_probs: torch.Tensor, num_experts: int) -> torch.Tensor:
    """L_aux = N * sum_i f_i * P_i (Switch Transformer)."""
    B = router_probs.size(0)
    if B == 0:
        return router_probs.sum() * 0.0
    hard = router_probs.argmax(dim=1)
    with torch.no_grad():
        f = torch.bincount(hard, minlength=num_experts).float() / float(B)
    P = router_probs.mean(dim=0)
    return num_experts * (f * P).sum()


def _routing_ratio_from_preds(preds: torch.Tensor, num_experts: int = 5) -> float:
    f = torch.bincount(preds, minlength=num_experts).float()
    frac = f / (f.sum() + 1e-9)
    return float(frac.max().item() / (frac.min().item() + 1e-9))


def _entropy_mean(probs: torch.Tensor) -> float:
    ent = -(probs * (probs.clamp_min(1e-9).log())).sum(dim=1)
    return float(ent.mean().item())


def _confusion_matrix(preds: torch.Tensor, targets: torch.Tensor, n_classes: int = 5) -> torch.Tensor:
    cm = torch.zeros((n_classes, n_classes), dtype=torch.long)
    for t, p in zip(targets.view(-1), preds.view(-1)):
        cm[t.long(), p.long()] += 1
    return cm


def set_train_router_mode(model: VisionRouter, mode: str = 'head_only', unfreeze_last_blocks: int = 2):
    """
    mode=head_only: entrena solo router_head.
    mode=partial_unfreeze: entrena router_head + últimos bloques del ViT.
    """
    for p in model.parameters():
        p.requires_grad = False

    for p in model.router_head.parameters():
        p.requires_grad = True

    if mode == 'partial_unfreeze':
        n = len(model.vit.blocks)
        start = max(0, n - int(unfreeze_last_blocks))
        for bi in range(start, n):
            for p in model.vit.blocks[bi].parameters():
                p.requires_grad = True


def build_optimizer_for_mode(model: VisionRouter, mode: str, lr_router: float, lr_vit: float):
    if mode == 'head_only':
        params = [p for p in model.router_head.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=lr_router, weight_decay=1e-4)

    router_params = [p for p in model.router_head.parameters() if p.requires_grad]
    vit_params = [p for n, p in model.named_parameters() if p.requires_grad and 'router_head' not in n]
    return torch.optim.AdamW(
        [
            {'params': router_params, 'lr': lr_router},
            {'params': vit_params, 'lr': lr_vit},
        ],
        weight_decay=1e-4,
    )


# Dataset IDs que usan etiqueta de tarea single-label (puede usarse CE directamente).
# NIH (0) es multi-label con espacio distinto al experto; se omite L_task para ese dominio.
_SINGLE_LABEL_DATASETS = {1, 2, 3, 4}  # ISIC, Osteo, LUNA16, Pancreas


def train_router_one_epoch(
    model: VisionRouter,
    dataloader,
    optimizer,
    device,
    *,
    alpha_aux: float = 0.05,
    label_smoothing: float = 0.0,
    scaler=None,
    max_batches: int | None = None,
    experts: dict | None = None,
):
    """
    Entrena el router una época.

    Flujo base (experts=None):
      L = CE(router_logits, expert_ids) + alpha_aux * L_aux

    Flujo profesor (experts dict proporcionado + dataloader con task labels):
      1. Router → gating probs → Top-1 expert elegido.
      2. Forward del experto (frozen, sin grad en pesos del experto).
      3. Logits_experto × gating_score → L_task (CE por muestra; crea gradiente
         hacia el router a través de gating_score).
      4. L = L_routing + L_task + alpha_aux * L_aux
         (expertos: requires_grad=False; router+cabeza: sí).
    """
    model.train()
    routing_criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    task_criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_ce = 0.0
    total_aux = 0.0
    n_samples = 0
    n_correct = 0

    all_preds = []
    all_targets = []
    all_entropy = []

    use_amp = scaler is not None and str(device).startswith('cuda')

    for bi, batch in enumerate(dataloader):
        if max_batches is not None and bi >= max_batches:
            break

        # Soporte batch con y sin etiquetas de tarea (2 vs 4 elementos)
        if len(batch) == 4:
            tensors, expert_ids, task_labels_raw, _ = batch
        else:
            tensors, expert_ids = batch
            task_labels_raw = None

        tensors = [t.to(device, non_blocking=True) for t in tensors]
        expert_ids = expert_ids.to(device, non_blocking=True).long()
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits_router, _ = model(tensors)
            probs = F.softmax(logits_router, dim=1)  # [B, 5]

            # --- L_routing: supervisión de enrutamiento (CE vs dataset de origen) ---
            L_routing = routing_criterion(logits_router, expert_ids)

            # --- Flujo del profesor: experto forward + L_task ---
            L_task = None
            if experts is not None and task_labels_raw is not None:
                pred_expert_ids = probs.argmax(dim=1)  # top-1 del router (0-4)
                task_parts = []
                for eid_val in pred_expert_ids.unique().tolist():
                    eid_val = int(eid_val)
                    if eid_val not in _SINGLE_LABEL_DATASETS:
                        continue  # NIH multi-label: espacio incompatible, se omite
                    expert_key = eid_val + 1  # dataset_id 0-4 → EXPERT_SPECS 1-5
                    if expert_key not in experts:
                        continue
                    mask = (pred_expert_ids == eid_val)
                    idxs = mask.nonzero(as_tuple=False).view(-1).tolist()
                    # Solo etiqueta escalar para CE (evita vec multi-label NIH si el router enruta mal)
                    valid_idxs = []
                    for i in idxs:
                        tlr = task_labels_raw[i]
                        if tlr is None:
                            continue
                        if isinstance(tlr, np.ndarray) and tlr.size != 1:
                            continue
                        if isinstance(tlr, torch.Tensor) and tlr.numel() != 1:
                            continue
                        valid_idxs.append(i)
                    if not valid_idxs:
                        continue
                    tl_list = []
                    for i in valid_idxs:
                        tlr = task_labels_raw[i]
                        if isinstance(tlr, torch.Tensor):
                            tl_list.append(int(tlr.detach().cpu().item()))
                        elif isinstance(tlr, np.ndarray):
                            tl_list.append(int(np.asarray(tlr).reshape(-1)[0]))
                        else:
                            tl_list.append(int(tlr))
                    tl_sub = torch.tensor(tl_list, dtype=torch.long, device=device)
                    # Forward del experto (congelado: sin grad en sus pesos)
                    x_sub = torch.stack([tensors[i] for i in valid_idxs])
                    with torch.no_grad():
                        expert_logits = experts[expert_key](x_sub)  # [n, n_expert_classes]
                    # Ponderar por gating score: único camino de gradiente hacia el router
                    g_scores = probs[
                        torch.tensor(valid_idxs, device=device), eid_val
                    ].unsqueeze(1)  # [n, 1]
                    weighted = expert_logits * g_scores  # [n, n_expert_classes]
                    L_task_i = task_criterion(weighted, tl_sub)
                    task_parts.append(L_task_i)
                if task_parts:
                    L_task = sum(task_parts) / len(task_parts)

            aux = _switch_aux_loss(probs, model.router_head.out_features)
            # Diagrama del profesor:
            #   Con expertos → L = L_task + α·L_aux  (el router aprende del diagnóstico real)
            #   Sin expertos / fallback (ej. NIH multi-label sin L_task) → L = L_routing + α·L_aux
            if L_task is not None:
                loss = L_task + alpha_aux * aux
            else:
                loss = L_routing + alpha_aux * aux

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        preds = logits_router.argmax(dim=1)
        bs = expert_ids.size(0)
        total_loss += loss.item() * bs
        total_ce += L_routing.item() * bs
        total_aux += aux.item() * bs
        n_samples += bs
        n_correct += (preds == expert_ids).sum().item()

        all_preds.append(preds.detach().cpu())
        all_targets.append(expert_ids.detach().cpu())
        all_entropy.append((-(probs * probs.clamp_min(1e-9).log()).sum(dim=1)).detach().cpu())

    if n_samples == 0:
        return {'loss': 0.0, 'ce': 0.0, 'aux': 0.0, 'routing_acc': 0.0, 'ratio': 999.0, 'entropy': 0.0, 'cm': torch.zeros((5, 5), dtype=torch.long), 'n': 0}

    preds_all = torch.cat(all_preds)
    targets_all = torch.cat(all_targets)
    ratio = _routing_ratio_from_preds(preds_all, num_experts=model.router_head.out_features)
    entropy = float(torch.cat(all_entropy).mean().item())
    cm = _confusion_matrix(preds_all, targets_all, n_classes=model.router_head.out_features)

    return {
        'loss': total_loss / n_samples,
        'ce': total_ce / n_samples,
        'aux': total_aux / n_samples,
        'routing_acc': n_correct / n_samples,
        'ratio': ratio,
        'entropy': entropy,
        'cm': cm,
        'n': n_samples,
    }


def eval_router_on_cls(router_head: nn.Module, Z_val_np: np.ndarray, y_val_np: np.ndarray, device: str = 'cpu'):
    router_head.eval()
    Z_t = torch.tensor(Z_val_np, dtype=torch.float32, device=device)
    y_t = torch.tensor(y_val_np, dtype=torch.long, device=device)
    with torch.no_grad():
        logits = router_head(Z_t)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        acc = (preds == y_t).float().mean().item()
    ratio = _routing_ratio_from_preds(preds.detach().cpu(), num_experts=probs.shape[1])
    entropy = _entropy_mean(probs.detach().cpu())
    cm = _confusion_matrix(preds.detach().cpu(), y_t.detach().cpu(), n_classes=probs.shape[1])
    return {'val_acc': float(acc), 'val_ratio': float(ratio), 'val_entropy': float(entropy), 'val_cm': cm}


def _save_router_ckpt(model, ckpt_path, mode, alpha_aux, history, best_meta, ep):
    """Guarda checkpoint completo del router (cabeza + ViT backbone)."""
    os.makedirs(os.path.dirname(ckpt_path) or '.', exist_ok=True)
    torch.save(
        {
            'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'router_head': {k: v.detach().cpu() for k, v in model.router_head.state_dict().items()},
            'best_epoch': ep,
            'best_meta': best_meta,
            'mode': mode,
            'alpha_aux': alpha_aux,
            'history': [{k: v for k, v in h.items() if k not in ('train_cm', 'val_cm')} for h in history],
        },
        ckpt_path,
    )


def fit_router_with_eval(
    model: VisionRouter,
    dataloader,
    Z_val_np: np.ndarray,
    y_val_np: np.ndarray,
    device: str,
    *,
    mode: str = 'head_only',
    epochs: int = 10,
    alpha_aux: float = 0.01,
    lr_router: float = 1e-3,
    lr_vit: float = 3e-5,
    label_smoothing: float = 0.1,
    unfreeze_last_blocks: int = 2,
    max_batches_per_epoch: int | None = None,
    ckpt_path: str | None = None,
    experts: dict | None = None,
):
    """
    Entrena el router con evaluación por época.

    experts: dict {expert_key: nn.Module} — si se proporciona, activa el flujo del
      profesor (forward del experto elegido + L_task). El dataloader debe haber sido
      construido con include_task_label=True para que los batches incluyan etiquetas de
      tarea. Sin experts, el entrenamiento usa solo L_routing + L_aux (flujo base).

    Guarda el mejor checkpoint en disco cada vez que val_score mejora (no solo al final).
    """
    set_train_router_mode(model, mode=mode, unfreeze_last_blocks=unfreeze_last_blocks)
    model.to(device)
    _dev_real = next(model.parameters()).device
    print(f"[fit_router_with_eval] pesos del router en: {_dev_real} (objetivo: {device})")
    if str(_dev_real) == 'cpu' and torch.cuda.is_available():
        print(
            "  AVISO: el modelo quedó en CPU aunque hay GPU. "
            "Pasa device='cuda' o torch.device('cuda') explícitamente."
        )
    if experts is not None:
        print(f"  Flujo del profesor activo: {len(experts)} expertos en forward de entrenamiento.")

    optimizer = build_optimizer_for_mode(model, mode=mode, lr_router=lr_router, lr_vit=lr_vit)
    use_cuda = str(device).startswith('cuda') and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=1e-5)

    history = []
    best_score = -1e9
    best_state = None

    for ep in range(1, epochs + 1):
        train_m = train_router_one_epoch(
            model, dataloader, optimizer, device,
            alpha_aux=alpha_aux,
            label_smoothing=label_smoothing,
            scaler=scaler if use_cuda else None,
            max_batches=max_batches_per_epoch,
            experts=experts,
        )
        val_m = eval_router_on_cls(model.router_head, Z_val_np, y_val_np, device=device)
        scheduler.step()

        vram_mb = 0.0
        if use_cuda:
            vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            torch.cuda.reset_peak_memory_stats()

        combined = {
            'epoch': ep,
            'mode': mode,
            'alpha_aux': alpha_aux,
            'lr_router': optimizer.param_groups[0]['lr'],
            'train_loss': train_m['loss'],
            'train_ce': train_m['ce'],
            'train_aux': train_m['aux'],
            'train_acc': train_m['routing_acc'],
            'train_ratio': train_m['ratio'],
            'train_entropy': train_m['entropy'],
            'val_acc': val_m['val_acc'],
            'val_ratio': val_m['val_ratio'],
            'val_entropy': val_m['val_entropy'],
            'vram_mb': vram_mb,
            'train_cm': train_m['cm'],
            'val_cm': val_m['val_cm'],
        }
        history.append(combined)

        print(
            f"[{mode}] Ep {ep}/{epochs}  "
            f"train_acc={combined['train_acc']:.4f} val_acc={combined['val_acc']:.4f}  "
            f"ratio={combined['val_ratio']:.3f} ent={combined['val_entropy']:.3f}  "
            f"aux={combined['train_aux']:.4f} vram_mb={combined['vram_mb']:.1f}"
        )

        # Prioriza val_acc y penaliza desbalance >1.30
        score = combined['val_acc'] - 0.02 * max(0.0, combined['val_ratio'] - 1.30)
        if score > best_score:
            best_score = score
            best_state = {
                'router_head': {k: v.detach().cpu() for k, v in model.router_head.state_dict().items()},
                'meta': combined,
            }
            # Guardar mejor checkpoint en disco en cuanto mejora (no solo al final)
            if ckpt_path:
                _save_router_ckpt(model, ckpt_path, mode, alpha_aux, history, combined, ep)
                print(f"  → mejor ckpt guardado ep {ep} (score={score:.4f}): {ckpt_path}")

    # Restaurar los mejores pesos al modelo en memoria
    if best_state is not None:
        model.router_head.load_state_dict(best_state['router_head'])

    return model, history


print('Funciones de entrenamiento listas: fit_router_with_eval + métricas de gating/balance/confusión.')

Funciones de entrenamiento listas: fit_router_with_eval + métricas de gating/balance/confusión.


In [17]:
def validate_cls_arrays(Z, y, name='Z', expected_dim=192):
    """Comprueba forma, NaN/Inf y estadísticas básicas de los CLS guardados."""
    Z = np.asarray(Z)
    y = np.asarray(y).ravel()
    print(f'=== {name} ===')
    print(f'  shape: {Z.shape}  dtype: {Z.dtype}')
    assert Z.ndim == 2, 'Z debe ser [N, d_model]'
    assert Z.shape[1] == expected_dim, f'Última dim debe ser {expected_dim}, got {Z.shape[1]}'
    assert len(y) == len(Z), 'y y Z deben tener el mismo N'

    n_bad = np.sum(~np.isfinite(Z))
    print(f'  NaN/Inf: {n_bad} / {Z.size}')
    if n_bad:
        print('  ⚠️ Corrige el extractor o datos antes del ablation.')

    print(f'  min/max: {np.nanmin(Z):.4f} / {np.nanmax(Z):.4f}')
    print(f'  mean/std (global): {np.mean(Z):.6f} / {np.std(Z):.6f}')

    for e in range(5):
        m = Z[y == e]
        if len(m) == 0:
            print(f'  experto {e}: sin muestras')
            continue
        norms = np.linalg.norm(m, axis=1)
        print(f'  experto {e}: n={len(m)}  ||CLS||_2 mean={norms.mean():.3f} std={norms.std():.3f}')
    print()
    return Z, y


def linear_probe_routing_accuracy(Z_train, y_train, Z_val, y_val, seed=42):
    """Clasificador lineal rápido: si >> 20%, los embeddings llevan señal de modalidad."""
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=2000, random_state=seed, multi_class='multinomial')
    clf.fit(Z_train, y_train)
    acc = float(clf.score(Z_val, y_val))
    print(f'Linear probe (routing): val accuracy = {acc:.4f}  (chance ≈ 0.20)')
    return acc


# --- Ejemplo: cargar arrays guardados y validar (ajusta SAVE_DIR) ---
# SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')
# Z_tr = np.load(os.path.join(SAVE_DIR, 'Z_train.npy'))
# y_tr = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
# Z_va = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'))
# y_va = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))
# validate_cls_arrays(Z_tr, y_tr, 'Z_train')
# validate_cls_arrays(Z_va, y_va, 'Z_val')
# linear_probe_routing_accuracy(Z_tr, y_tr, Z_va, y_va)

print('Funciones listas: validate_cls_arrays, linear_probe_routing_accuracy')

Funciones listas: validate_cls_arrays, linear_probe_routing_accuracy


In [18]:
def verify_cls_matches_model(
    model,
    preprocessor,
    file_paths,
    Z_reference_rows,
    device='cpu',
    rtol=1e-4,
    atol=1e-5,
):
    """
    Compara CLS recalculado en vivo con filas de Z ya guardadas (mismo orden de file_paths).
    Usa el mismo model.eval() y sin grad. Útil para depurar un subconjunto pequeño.
    """
    model.eval()
    model.to(device)
    diffs = []
    with torch.no_grad():
        for i, path in enumerate(file_paths):
            t = preprocessor(path).unsqueeze(0)
            # El forward del VisionRouter espera lista de tensores por muestra
            _, cls_live = model([t.squeeze(0)])
            v_live = cls_live.cpu().numpy().reshape(-1)
            v_ref = np.asarray(Z_reference_rows[i]).reshape(-1)
            err = np.max(np.abs(v_live - v_ref))
            diffs.append(err)
            ok = np.allclose(v_live, v_ref, rtol=rtol, atol=atol)
            print(f'  [{i}] {os.path.basename(path)}  max|Δ|={err:.2e}  ok={ok}')
    print(f'Máximo error global: {max(diffs):.2e}')
    return diffs


# Ejemplo (descomenta; necesitas mismos pesos que al extraer):
# paths = [all_files[0], all_files[100], ...][:5]
# Z_rows = Z_train[[0, 10, 20, 30, 40]]  # mismas filas que paths
# verify_cls_matches_model(router, preprocessor, paths, Z_rows, device=device)

## PASO 0 — Cargar Expertos Pre-entrenados (congelados)

**Antes:** ejecuta la celda de **arquitecturas embebidas** (justo después de Fase 0) para definir `load_all_experts_from_drive`.

Los pesos se leen de `WEIGHTS_DIR` / `CKPT_DIR` en Drive. Las definiciones de modelos están **en el mismo notebook** (sin archivo `.py` externo).

Se cargan con `requires_grad=False`: **no reciben gradiente en ninguna fase del router**.

> El router no entrena los expertos: los usa como función fija.

In [19]:
import os
import torch

# Misma carpeta que WEIGHTS_DIR (Fase 0). Checkpoints opcionales para referencia.
CKPT_DIR = WEIGHTS_DIR if "WEIGHTS_DIR" in globals() else "/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/"
CKPT_PATHS = {
    "exp1_nih": os.path.join(CKPT_DIR, "Experts_2D", "MaxViT_NIH_5cls.pth"),
    "exp2_isic": os.path.join(CKPT_DIR, "exp2_ISIC_EfficientNetB3_best.pth"),
    "exp3_osteo": os.path.join(CKPT_DIR, "exp3_Osteo_VGG16BN_best.pth"),
    "exp4_luna16": os.path.join(CKPT_DIR, "exp4_LUNA16_3D_best.pth"),
    "exp5_pancreas": os.path.join(CKPT_DIR, "exp5_Pancreas_3D_best.pth"),
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
print("CKPT_DIR:", CKPT_DIR)


Dispositivo: cpu
CKPT_DIR: /content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/


## PASO 1 — Extracción de CLS Tokens (Fase 0 del Ablation Study)

Pasamos **~1 000 imágenes por dataset** (5 000 total) a través del backbone ViT-Tiny
pre-entrenado de timm y guardamos los vectores `[N, 192]` en Drive.

- La **etiqueta** es el `dataset_id` (0=NIH, 1=ISIC, 2=Osteo, 3=LUNA16, 4=Páncreas),
  **no** la etiqueta de enfermedad.
- Split 80/20 estratificado → `Z_train`, `Z_val`, `y_train_expert`, `y_val_expert`.
- Una vez guardados, el ablation study se ejecuta en segundos sin reprocesar imágenes.

In [24]:
# ── Construir DataLoader Mixto balanceado (1 000 por dataset) ────────────────
loader_cls = build_router_dataloader(
    roots=DATASET_ROOTS,
    preprocessor=preprocessor,
    batch_size=16,
    num_workers=2,
    sample_cap=1000,
)
print('DataLoader CLS listo.')


  [NIH] Balanceado: reducido a 1000 muestras (ID=0)
  [ISIC] Balanceado: reducido a 1000 muestras (ID=1)
  [Osteo] Balanceado: reducido a 1000 muestras (ID=2)
  [LUNA16] 889 archivos cargados en total (ID=3)
  [Pancreas] 557 archivos cargados en total (ID=4)

DataLoader listo: 4446 muestras, batch_size=16, include_task_label=False
DataLoader CLS listo.


## PASO 2 — Ablation Study: 4 Mecanismos de Routing

Se comparan sobre los **mismos CLS tokens** guardados en PASO 1.
Los mecanismos B, C y D corren en **CPU** (sin GPU).

| # | Mecanismo | Tipo | Gradiente |
|---|-----------|------|-----------|
| A | Linear + Softmax | Param (DL) | ✅ Sí |
| B | GMM (sklearn) | Param (EM) | ❌ No |
| C | Naive Bayes (sklearn) | Param (MLE) | ❌ No |
| D | k-NN FAISS coseno | No param | ❌ No |

In [27]:
# ── Cargar CLS desde disco (si el kernel se reinicia) ────────────────────────
import numpy as np
from collections import Counter

SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')  # ya definido arriba

Z_train_np = np.load(os.path.join(SAVE_DIR, 'Z_train.npy')).astype('float32')
Z_val_np   = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'  )).astype('float32')
y_train_np = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
y_val_np   = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))

print(f'Z_train: {Z_train_np.shape}  y_train: {dict(Counter(y_train_np))}')
print(f'Z_val:   {Z_val_np.shape}   y_val:   {dict(Counter(y_val_np))}')


Z_train: (3556, 192)  y_train: {np.int64(1): 800, np.int64(4): 445, np.int64(2): 800, np.int64(0): 800, np.int64(3): 711}
Z_val:   (890, 192)   y_val:   {np.int64(3): 178, np.int64(4): 112, np.int64(1): 200, np.int64(2): 200, np.int64(0): 200}


### Mecanismo A — ViT + Linear + Softmax (Baseline DL)

Entrena `nn.Linear(192→5)` con CrossEntropyLoss + `L_aux` (Switch Transformer).
Es el **único mecanismo con gradiente** y el que tiene penalización de balance de carga.

In [28]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader as TorchDL

# ── Tensores ─────────────────────────────────────────────────────────────────
X_tr = torch.tensor(Z_train_np)
X_vl = torch.tensor(Z_val_np)
y_tr = torch.tensor(y_train_np, dtype=torch.long)
y_vl = torch.tensor(y_val_np,   dtype=torch.long)

# ── Modelo: cabeza lineal aislada ────────────────────────────────────────────
linear_head_A = nn.Linear(192, 5)
optimizer_A   = torch.optim.Adam(linear_head_A.parameters(), lr=1e-3)
criterion_A   = nn.CrossEntropyLoss()
EPOCHS_A      = 50
ALPHA_AUX     = 0.01

# Dataloader mini-batch para evitar OOM en CPU si Z es grande
ds_tr = TensorDataset(X_tr, y_tr)
dl_tr = TorchDL(ds_tr, batch_size=256, shuffle=True)

history_A = []
for epoch in range(1, EPOCHS_A + 1):
    linear_head_A.train()
    ep_loss, ep_correct, ep_n = 0.0, 0, 0
    for xb, yb in dl_tr:
        logits = linear_head_A(xb)
        probs  = F.softmax(logits, dim=-1)
        # L_aux Switch Transformer
        hard = probs.argmax(dim=1)
        f_i  = torch.bincount(hard, minlength=5).float() / xb.size(0)
        P_i  = probs.mean(dim=0)
        l_aux = 5 * (f_i * P_i).sum()
        l_ce  = criterion_A(logits, yb)
        loss  = l_ce + ALPHA_AUX * l_aux
        optimizer_A.zero_grad(); loss.backward(); optimizer_A.step()
        ep_loss += loss.item() * xb.size(0)
        ep_correct += (logits.argmax(1) == yb).sum().item()
        ep_n += xb.size(0)
    train_acc = ep_correct / ep_n
    # validación
    linear_head_A.eval()
    with torch.no_grad():
        val_preds = linear_head_A(X_vl).argmax(dim=1)
    val_acc = (val_preds == y_vl).float().mean().item()
    history_A.append({'epoch': epoch, 'train_acc': train_acc, 'val_acc': val_acc,
                       'loss': ep_loss / ep_n})
    if epoch % 10 == 0 or epoch == 1:
        print(f'  Época {epoch:3d}/{EPOCHS_A}  loss={ep_loss/ep_n:.4f}  '
              f'train_acc={train_acc:.4f}  val_acc={val_acc:.4f}')

acc_A = val_acc
print(f'\n✅ Mecanismo A — Routing Accuracy (val): {acc_A:.4f}')


  Época   1/50  loss=1.4427  train_acc=0.3701  val_acc=0.4753
  Época  10/50  loss=0.5674  train_acc=0.8383  val_acc=0.8326
  Época  20/50  loss=0.4668  train_acc=0.8743  val_acc=0.8539
  Época  30/50  loss=0.4040  train_acc=0.8923  val_acc=0.8753
  Época  40/50  loss=0.3586  train_acc=0.9072  val_acc=0.8876
  Época  50/50  loss=0.3247  train_acc=0.9168  val_acc=0.9056

✅ Mecanismo A — Routing Accuracy (val): 0.9056


In [30]:
# ── Balance de carga del Mecanismo A ─────────────────────────────────────────
linear_head_A.eval()
with torch.no_grad():
    all_preds_A = linear_head_A(X_vl).argmax(dim=1).numpy()
f_counts_A = np.bincount(all_preds_A, minlength=5)
f_frac_A   = f_counts_A / f_counts_A.sum()
ratio_A    = f_frac_A.max() / (f_frac_A.min() + 1e-9)
print('Balance de carga — Mecanismo A (val):')
for i, (cnt, frac) in enumerate(zip(f_counts_A, f_frac_A)):
    print(f'  Experto {i}: {cnt:4d} muestras  ({frac*100:.1f}%)')
print(f'  max(f_i)/min(f_i) = {ratio_A:.3f}')
if ratio_A > 1.30:
    print('  ⚠️  RATIO > 1.30 — aumentar alpha_aux o más épocas')
else:
    print('  ✅ Balance dentro del umbral (< 1.30)')


Balance de carga — Mecanismo A (val):
  Experto 0:  236 muestras  (26.5%)
  Experto 1:  171 muestras  (19.2%)
  Experto 2:  194 muestras  (21.8%)
  Experto 3:  177 muestras  (19.9%)
  Experto 4:  112 muestras  (12.6%)
  max(f_i)/min(f_i) = 2.107
  ⚠️  RATIO > 1.30 — aumentar alpha_aux o más épocas


### Mecanismo B — ViT + GMM (Paramétrico-EM)

Gaussian Mixture Model con 5 componentes. Entrenado con EM sobre `Z_train`.
Sin gradiente descendente. Corre completamente en CPU.

In [31]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score

# --- Covariance type: 'full' si N_por_experto >= 1000, 'diag' si es menor ---
min_n_exp = min(Counter(y_train_np).values())
cov_type = 'full' if min_n_exp >= 1000 else 'diag'
print(f'Mínimo muestras por experto en train: {min_n_exp} → covariance_type="{cov_type}"')

gmm = GaussianMixture(
    n_components=5,
    covariance_type=cov_type,
    max_iter=200,
    random_state=42,
    verbose=0,
)
print('Ajustando GMM (EM)...')
gmm.fit(Z_train_np)

probs_gmm_val   = gmm.predict_proba(Z_val_np)
routing_pred_B  = probs_gmm_val.argmax(axis=1)
acc_B = accuracy_score(y_val_np, routing_pred_B)
print(f'✅ Mecanismo B (GMM) — Routing Accuracy (val): {acc_B:.4f}')

# Balance de carga
f_counts_B = np.bincount(routing_pred_B, minlength=5)
f_frac_B   = f_counts_B / f_counts_B.sum()
ratio_B    = f_frac_B.max() / (f_frac_B.min() + 1e-9)
print(f'  Balance max(f_i)/min(f_i) = {ratio_B:.3f}')


Mínimo muestras por experto en train: 445 → covariance_type="diag"
Ajustando GMM (EM)...
✅ Mecanismo B (GMM) — Routing Accuracy (val): 0.1573
  Balance max(f_i)/min(f_i) = 27.400


### Mecanismo C — ViT + Naive Bayes (Paramétrico-MLE)

Gaussian Naive Bayes: asume independencia entre las 192 dimensiones del CLS token.
Solución analítica en forma cerrada. El más rápido de los 4 mecanismos.

In [32]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
print('Ajustando Naive Bayes (MLE)...')
nb.fit(Z_train_np, y_train_np)

probs_nb_val    = nb.predict_proba(Z_val_np)
routing_pred_C  = probs_nb_val.argmax(axis=1)
acc_C = accuracy_score(y_val_np, routing_pred_C)
print(f'✅ Mecanismo C (Naive Bayes) — Routing Accuracy (val): {acc_C:.4f}')

# Balance de carga
f_counts_C = np.bincount(routing_pred_C, minlength=5)
f_frac_C   = f_counts_C / f_counts_C.sum()
ratio_C    = f_frac_C.max() / (f_frac_C.min() + 1e-9)
print(f'  Balance max(f_i)/min(f_i) = {ratio_C:.3f}')


Ajustando Naive Bayes (MLE)...
✅ Mecanismo C (Naive Bayes) — Routing Accuracy (val): 0.8258
  Balance max(f_i)/min(f_i) = 2.534


### Mecanismo D — ViT + k-NN (FAISS, distancia coseno)

El único método **no paramétrico**: no asume forma distribucional.
Su 'memoria' son los N_train embeddings guardados.
Usa FAISS con normalización L2 (equivale a similitud coseno).

In [33]:
!pip install -q faiss-cpu
import faiss

d = Z_train_np.shape[1]  # 192
Z_tr_f = Z_train_np.copy()
Z_vl_f = Z_val_np.copy()
faiss.normalize_L2(Z_tr_f)  # in-place L2 normalization → coseno
faiss.normalize_L2(Z_vl_f)

index = faiss.IndexFlatIP(d)  # Inner Product sobre vectores L2-norm = coseno
index.add(Z_tr_f)
print(f'Índice FAISS construido: {index.ntotal} vectores en d={d}')

K = 5
D_mat, I_mat = index.search(Z_vl_f, K)
neighbors_labels = y_train_np[I_mat]  # [N_val, K]
routing_pred_D = np.apply_along_axis(
    lambda x: np.bincount(x, minlength=5).argmax(),
    axis=1,
    arr=neighbors_labels.astype(int),
)
acc_D = accuracy_score(y_val_np, routing_pred_D)
print(f'✅ Mecanismo D (k-NN k={K} FAISS) — Routing Accuracy (val): {acc_D:.4f}')

# Balance de carga
f_counts_D = np.bincount(routing_pred_D, minlength=5)
f_frac_D   = f_counts_D / f_counts_D.sum()
ratio_D    = f_frac_D.max() / (f_frac_D.min() + 1e-9)
print(f'  Balance max(f_i)/min(f_i) = {ratio_D:.3f}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.8 MB/s eta 0:00:00
Índice FAISS construido: 3556 vectores en d=192
✅ Mecanismo D (k-NN k=5 FAISS) — Routing Accuracy (val): 0.9629
  Balance max(f_i)/min(f_i) = 1.884


## PASO 2 — Tabla Comparativa del Ablation Study

Comparamos los 4 mecanismos sobre exactamente el mismo backbone ViT congelado.
El experimento es justo: solo varía la cabeza de decisión.

In [35]:
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    import faiss
    _HAS_FAISS = True
except Exception:
    _HAS_FAISS = False

# Asegurar que arrays CLS estén disponibles
if 'Z_train_np' not in globals():
    SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')
    Z_train_np = np.load(os.path.join(SAVE_DIR, 'Z_train.npy'))
    Z_val_np = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'))
    y_train_np = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
    y_val_np = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))

# A) ViT + Linear (router_head entrenado)
router.eval()
with torch.no_grad():
    logits_A = router.router_head(torch.tensor(Z_val_np, dtype=torch.float32, device=DEVICE))
    pred_A = logits_A.argmax(dim=1).cpu().numpy()
acc_A = float(accuracy_score(y_val_np, pred_A))
ratio_A = _routing_ratio_from_preds(torch.tensor(pred_A), num_experts=5)

# B) GMM
gmm = GaussianMixture(n_components=5, covariance_type='full', max_iter=200, random_state=42)
gmm.fit(Z_train_np)
pred_B = gmm.predict_proba(Z_val_np).argmax(axis=1)
acc_B = float(accuracy_score(y_val_np, pred_B))
ratio_B = _routing_ratio_from_preds(torch.tensor(pred_B), num_experts=5)

# C) Naive Bayes
nb = GaussianNB()
nb.fit(Z_train_np, y_train_np)
pred_C = nb.predict_proba(Z_val_np).argmax(axis=1)
acc_C = float(accuracy_score(y_val_np, pred_C))
ratio_C = _routing_ratio_from_preds(torch.tensor(pred_C), num_experts=5)

# D) k-NN (FAISS o fallback sklearn)
if _HAS_FAISS:
    Z_train_f = Z_train_np.astype('float32').copy()
    Z_val_f = Z_val_np.astype('float32').copy()
    faiss.normalize_L2(Z_train_f)
    faiss.normalize_L2(Z_val_f)
    index = faiss.IndexFlatIP(Z_train_f.shape[1])
    index.add(Z_train_f)
    _, I = index.search(Z_val_f, k=5)
    pred_D = np.apply_along_axis(lambda x: np.bincount(x, minlength=5).argmax(), 1, y_train_np[I])
else:
    from sklearn.neighbors import KNeighborsClassifier
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(Z_train_np, y_train_np)
    pred_D = knn.predict(Z_val_np)

acc_D = float(accuracy_score(y_val_np, pred_D))
ratio_D = _routing_ratio_from_preds(torch.tensor(pred_D), num_experts=5)

ablation_results_raw = pd.DataFrame([
    {'Router': 'A: ViT + Linear (DL)', 'Tipo': 'Param (gradiente)', 'Routing_Acc': acc_A, 'Balance_ratio': ratio_A, 'Gradiente': 'Si', 'L_aux': 'Si'},
    {'Router': 'B: ViT + GMM', 'Tipo': 'Param (EM)', 'Routing_Acc': acc_B, 'Balance_ratio': ratio_B, 'Gradiente': 'No', 'L_aux': 'No'},
    {'Router': 'C: ViT + Naive Bayes', 'Tipo': 'Param (MLE)', 'Routing_Acc': acc_C, 'Balance_ratio': ratio_C, 'Gradiente': 'No', 'L_aux': 'No'},
    {'Router': 'D: ViT + k-NN (FAISS)', 'Tipo': 'No param', 'Routing_Acc': acc_D, 'Balance_ratio': ratio_D, 'Gradiente': 'No', 'L_aux': 'No'},
]).sort_values('Routing_Acc', ascending=False)

print('=' * 80)
print('TABLA ABLATION STUDY - Routing Mechanisms')
print('=' * 80)
print(ablation_results_raw.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('=' * 80)

ablation_results_raw.to_csv(os.path.join(FEATURE_DIR, 'ablation_results.csv'), index=False)
print('Tabla guardada en Drive: ablation_results.csv')

winner = ablation_results_raw.iloc[0]
print(f"\nRouter ganador: {winner['Router']} | acc={winner['Routing_Acc']:.4f}")


## PASO 3 — Training Loop del Router (Fase 2 — Método B)

Con los expertos **congelados**, se entrena solo el router head usando el
`fit_router_head()` ya implementado en Cell 24.

**Lógica:**
```
imagen → VisionRouter → probs [B,5] → Top-1 → expert_i (frozen) → logits
L_total = CrossEntropy(logits_router, y_expert) + 0.01 × L_aux
backward() → SOLO router_head se actualiza
```

> Independientemente del ganador del ablation, el entrenamiento sobre el DataLoader
> real siempre usa el VisionRouter con ViT+Linear para poder medir el balance de carga
> y la L_aux (consigna §5).

In [36]:
# --- Baseline reproducible + DataLoader balanceado (audit) ---
from torch.utils.data import DataLoader, WeightedRandomSampler

set_seed(42)


def build_router_dataloader_weighted(
    roots,
    preprocessor,
    *,
    batch_size=8,
    num_workers=2,
    sample_cap=1000,
    include_task_label: bool = False,
):
    """Loader balanceado por WeightedRandomSampler.

    include_task_label=True: el batch devuelve (tensors, expert_ids, task_labels, paths)
    lo que activa el flujo del profesor en train_router_one_epoch.
    """
    all_files, all_ids = [], []
    for name, (path, did) in roots.items():
        if name == 'LUNA16':
            cands = label_index.get('luna16', [])
            if sample_cap and len(cands) > sample_cap: random.seed(42); cands = random.sample(cands, sample_cap)
            all_files.extend(cands); all_ids.extend([did] * len(cands)); print(f'[LUNA16] n={len(cands)} Candidates (id={did})'); continue
        files = scan_dataset_files(path)
        if sample_cap is not None and len(files) > sample_cap:
            random.seed(42)
            files = random.sample(files, sample_cap)
        all_files.extend(files)
        all_ids.extend([did] * len(files))
        print(f'[{name}] n={len(files)} (id={did})')

    label_index = build_dataset_label_index(roots) if include_task_label else None
    dataset = MixedMedicalDataset(
        all_files, all_ids, preprocessor,
        label_index=label_index,
        include_task_label=include_task_label,
    )
    counts = Counter(all_ids)
    weights = [1.0 / counts[did] for did in all_ids]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(all_ids),
        replacement=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=mixed_collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )
    print(f'Loader balanceado listo: N={len(dataset)}, batch_size={batch_size}, task_labels={include_task_label}')
    print(f'Conteo por dominio: {dict(counts)}')
    return loader, counts


# Config rápida para barridos iniciales
BATCH_SIZE = 8
SAMPLE_CAP = 1000
NUM_WORKERS = 2

loader_train_router, domain_counts = build_router_dataloader_weighted(
    roots=DATASET_ROOTS,
    preprocessor=preprocessor,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    sample_cap=SAMPLE_CAP,
)

# Cargar split fijo de validación CLS para evaluación consistente
SAVE_DIR = os.path.join(FEATURE_DIR, 'ablation_data_v1')
Z_train_np = np.load(os.path.join(SAVE_DIR, 'Z_train.npy'))
Z_val_np = np.load(os.path.join(SAVE_DIR, 'Z_val.npy'))
y_train_np = np.load(os.path.join(SAVE_DIR, 'y_train_expert.npy'))
y_val_np = np.load(os.path.join(SAVE_DIR, 'y_val_expert.npy'))

print(f'CLS train: {Z_train_np.shape} | CLS val: {Z_val_np.shape}')
print('Baseline audit completado (seed/split/sample_cap fijos).')


[NIH] n=1000 (id=0)
[ISIC] n=1000 (id=1)
[Osteo] n=1000 (id=2)
[LUNA16] n=889 (id=3)
[Pancreas] n=557 (id=4)
Loader balanceado listo: N=4446, batch_size=8
Conteo por dominio: {0: 1000, 1: 1000, 2: 1000, 3: 889, 4: 557}
CLS train: (3556, 192) | CLS val: (890, 192)
Baseline audit completado (seed/split/sample_cap fijos).


In [37]:
# --- Protocolo completo: warm-up sweep + partial unfreeze ---
# Nota: para ejecución larga en Colab, ajusta épocas y max_batches_per_epoch.
# GPU: solo se usa si torch.cuda.is_available(). Sin GPU en Colab → Runtime →
# Cambiar tipo de entorno → Acelerador: GPU → Guardar y REINICIAR; luego re-ejecutar.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[device] DEVICE={DEVICE} | cuda.is_available()={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"         GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "         Sin CUDA: el router entrenará en CPU (lento). "
        "Activa GPU en Colab y reinicia el runtime si esperabas aceleración."
    )
# assert torch.cuda.is_available()  # descomenta para fallar en CPU y no seguir en silencio

WARMUP_EPOCHS = 10
PARTIAL_EPOCHS = 6
MAX_BATCHES_PER_EPOCH = None   # usar int (p.ej., 100) para pruebas rápidas

warmup_alphas = [0.01, 0.03, 0.05]
warmup_results = []
warmup_histories = {}

for alpha in warmup_alphas:
    print('\n' + '=' * 80)
    print(f'WARM-UP head_only con alpha_aux={alpha:.2f}')
    print('=' * 80)

    router_w = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=True)
    router_w, hist = fit_router_with_eval(
        model=router_w,
        dataloader=loader_train_router,
        Z_val_np=Z_val_np,
        y_val_np=y_val_np,
        device=DEVICE,
        mode='head_only',
        epochs=WARMUP_EPOCHS,
        alpha_aux=alpha,
        lr_router=1e-3,
        lr_vit=3e-5,
        label_smoothing=0.1,
        unfreeze_last_blocks=0,
        max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
        ckpt_path=os.path.join(FEATURE_DIR, f'router_head_warmup_alpha_{alpha:.2f}.pth'),
    )

    best_ep = max(hist, key=lambda x: x['val_acc'] - 0.02 * max(0.0, x['val_ratio'] - 1.30))
    warmup_results.append(
        {
            'alpha_aux': alpha,
            'best_val_acc': best_ep['val_acc'],
            'best_val_ratio': best_ep['val_ratio'],
            'best_val_entropy': best_ep['val_entropy'],
            'best_vram_mb': max(h['vram_mb'] for h in hist),
        }
    )
    warmup_histories[alpha] = hist

# Selección por mejor trade-off
warmup_results = sorted(
    warmup_results,
    key=lambda r: (r['best_val_acc'] - 0.02 * max(0.0, r['best_val_ratio'] - 1.30)),
    reverse=True,
)

best_alpha = warmup_results[0]['alpha_aux']
print('\nResultados warm-up (ordenados):')
for r in warmup_results:
    print(r)
print(f'\nAlpha seleccionado para fine-tuning parcial: {best_alpha:.2f}')

# Entrenamiento final con descongelado parcial de últimos bloques
router = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=True)
router, history_router = fit_router_with_eval(
    model=router,
    dataloader=loader_train_router,
    Z_val_np=Z_val_np,
    y_val_np=y_val_np,
    device=DEVICE,
    mode='partial_unfreeze',
    epochs=PARTIAL_EPOCHS,
    alpha_aux=best_alpha,
    lr_router=3e-4,
    lr_vit=3e-5,
    label_smoothing=0.1,
    unfreeze_last_blocks=2,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
    ckpt_path=os.path.join(FEATURE_DIR, 'router_head_phase2_best.pth'),
)

print('\nEntrenamiento completo del router finalizado (warm-up + partial unfreeze).')


In [ ]:
# --- MoE: todos los expertos fijados en VRAM (rápido; sin CPU↔GPU por batch) ---
# Carga los 5 modelos una vez en el mismo device. Requiere VRAM suficiente; si falla OOM, reduce batch o usa CPU.

import warnings


def _resolve_moe_device(device=None):
    if device is not None:
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    warnings.warn("CUDA no disponible: expertos permanecerán en CPU.")
    return torch.device("cpu")


def _vram_gib() -> float | None:
    if not torch.cuda.is_available():
        return None
    return torch.cuda.memory_allocated() / (1024 ** 3)


class ExpertsPinnedGPU:
    """
    Todos los expertos residen en `device` (idealmente CUDA) desde el inicio.
    `get(eid)` solo indexa en memoria — sin eviction ni `.to()` por muestra.
    """

    def __init__(self, expert_models: dict, device=None, *, verbose: bool = True):
        self.device = _resolve_moe_device(device)
        self.experts: dict[int, torch.nn.Module] = {}
        before = _vram_gib()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
            before = _vram_gib()

        for eid, model in sorted(expert_models.items(), key=lambda kv: int(kv[0])):
            eid = int(eid)
            try:
                model = model.to(self.device).eval()
            except RuntimeError as err:
                if "out of memory" in str(err).lower() and self.device.type == "cuda":
                    torch.cuda.empty_cache()
                    raise RuntimeError(
                        "OOM al subir expertos a GPU. Prueba: reduce batch en inferencia, "
                        "cierra otros procesos en la GPU, o ExpertsPinnedGPU(..., device='cpu')."
                    ) from err
                raise
            for p in model.parameters():
                p.requires_grad = False
            self.experts[eid] = model

        if self.device.type == "cuda":
            torch.cuda.synchronize()
        after = _vram_gib()
        if verbose:
            if after is not None and before is not None:
                print(
                    f"ExpertsPinnedGPU: {len(self.experts)} modelos en {self.device} | "
                    f"VRAM delta ~{after - before:.2f} GiB (aprox., solo expertos)"
                )
            else:
                print(f"ExpertsPinnedGPU: {len(self.experts)} modelos en {self.device}")

    def get(self, expert_id: int) -> torch.nn.Module:
        return self.experts[int(expert_id)]


def moe_forward_top1(router_model, experts_manager: ExpertsPinnedGPU, batch_tensors, device=None):
    """
    Inferencia Top-1:
      1) router → probs [B,5]
      2) agrupa por expert_id predicho
      3) forward del experto ya en VRAM (sin transferencias de modelo por batch)
    """
    dev = torch.device(device) if device is not None else experts_manager.device
    with torch.no_grad():
        logits, _ = router_model([t.to(dev, non_blocking=True) for t in batch_tensors])
        probs = F.softmax(logits, dim=1)
        pred_expert = probs.argmax(dim=1)

    outputs = [None] * len(batch_tensors)
    for eid in pred_expert.unique().tolist():
        idxs = (pred_expert == eid).nonzero(as_tuple=False).view(-1).tolist()
        expert = experts_manager.get(eid)
        x_sub = [batch_tensors[i].to(dev, non_blocking=True) for i in idxs]

        # Nota: cada experto puede requerir su propio collate/input shape.
        # Este bloque es un placeholder para integrar cada forward real.
        for k, i in enumerate(idxs):
            outputs[i] = expert(x_sub[k].unsqueeze(0))

    return outputs, probs, pred_expert


# Alias retrocompatible con código que aún importe el nombre antiguo
ExpertOnDemandManager = ExpertsPinnedGPU
moe_forward_top1_on_demand = moe_forward_top1

print("ExpertsPinnedGPU + moe_forward_top1 listos (todos los expertos en VRAM si hay CUDA).")

## Fase 3 (opcional) — Flujo del profesor: `L_task` + expertos congelados

Después del warm-up + partial unfreeze, puedes entrenar el router con **feedback de diagnóstico**:

- Carga los **5 expertos** (checkpoints en `WEIGHTS_DIR`) y fíjalos en VRAM con `ExpertsPinnedGPU`.
- Reconstruye el DataLoader con **`include_task_label=True`** (etiquetas de tarea por dataset).
- `fit_router_with_eval(..., experts=...)` usa **`L = L_task + α·L_aux`** cuando hay `L_task` (logits experto × gating score vs etiqueta real).
- **VRAM:** router + 5 expertos es pesado; baja `PROF_BATCH_SIZE` si hay OOM.

> Ejecuta **después** de la celda de `ExpertsPinnedGPU` y de haber corrido el protocolo warm-up (para tener `router_head_phase2_best.pth`).

In [ ]:
# --- Fase 3: entrenamiento con feedback de expertos (diagrama del profesor) ---
# Requiere: WEIGHTS_DIR, FEATURE_DIR, preprocessor, DATASET_ROOTS, Z_val_np, y_val_np, DEVICE, BATCH_SIZE, SAMPLE_CAP, NUM_WORKERS

RUN_PROFESSOR_MOE_PHASE = True  # pon False para omitir
PROF_BATCH_SIZE = min(4, BATCH_SIZE)  # bajar a 2 si OOM (router + 5 expertos)
PROFESSOR_EPOCHS = 8
PROF_MAX_BATCHES = None  # p.ej. 50 para prueba rápida

if not RUN_PROFESSOR_MOE_PHASE:
    print("Fase 3 (profesor) omitida (RUN_PROFESSOR_MOE_PHASE=False).")
else:
    experts_dict, expert_info = load_all_experts_from_drive(
        WEIGHTS_DIR, device="cpu", strict=False, pretrained_backbone=False
    )
    print_expert_load_report(expert_info)

    experts_mgr = ExpertsPinnedGPU(experts_dict, device=str(DEVICE))

    set_seed(42)
    loader_train_prof, _ = build_router_dataloader_weighted(
        roots=DATASET_ROOTS,
        preprocessor=preprocessor,
        batch_size=PROF_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        sample_cap=SAMPLE_CAP,
        include_task_label=True,
    )

    router_prof = VisionRouter(embed_dim=192, num_experts=5, num_layers=12, pretrained=True)
    ckpt_phase2 = os.path.join(FEATURE_DIR, "router_head_phase2_best.pth")
    if os.path.isfile(ckpt_phase2):
        sd = torch.load(ckpt_phase2, map_location=DEVICE)
        if "model_state_dict" in sd:
            router_prof.load_state_dict(sd["model_state_dict"], strict=False)
            print(f"Router cargado desde {ckpt_phase2} (model_state_dict completo).")
        elif "router_head" in sd:
            router_prof.router_head.load_state_dict(sd["router_head"], strict=True)
            print(f"Router: solo router_head desde {ckpt_phase2} (ViT ImageNet).")
    else:
        print(f"ADVERTENCIA: no existe {ckpt_phase2}; router desde pesos timm (sin warm-up).")

    router_prof, history_prof = fit_router_with_eval(
        model=router_prof,
        dataloader=loader_train_prof,
        Z_val_np=Z_val_np,
        y_val_np=y_val_np,
        device=DEVICE,
        mode="partial_unfreeze",
        epochs=PROFESSOR_EPOCHS,
        alpha_aux=globals().get("best_alpha", 0.03),
        lr_router=2e-4,
        lr_vit=2e-5,
        label_smoothing=0.1,
        unfreeze_last_blocks=2,
        max_batches_per_epoch=PROF_MAX_BATCHES,
        ckpt_path=os.path.join(FEATURE_DIR, "router_professor_Ltask_best.pth"),
        experts=experts_mgr.experts,
    )

    router = router_prof  # reemplaza referencia global para PASO 4 / evaluación
    history_router = history_prof
    print("Fase 3 (profesor) finalizada. Checkpoint: router_professor_Ltask_best.pth")


## PASO 4 — Evaluación Final y Balance de Carga

Medimos el **Routing Accuracy** del router entrenado sobre el conjunto de validación
y verificamos que `max(f_i)/min(f_i) < 1.30` (umbral de la consigna).

> ⚠️ Si el ratio supera 1.30 al momento de entrega → penalización del 40% sobre la nota.

In [ ]:
# --- Evaluación final del router (KPI: acc, balance, OOD, VRAM) ---
from sklearn.metrics import roc_auc_score

router.eval()
Z_val_t = torch.tensor(Z_val_np, dtype=torch.float32, device=DEVICE)
y_val_t = torch.tensor(y_val_np, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    final_logits = router.router_head(Z_val_t)
    final_probs = F.softmax(final_logits, dim=-1)
    final_preds = final_logits.argmax(dim=1)

final_acc = float((final_preds == y_val_t).float().mean().item())
ratio_final = _routing_ratio_from_preds(final_preds.detach().cpu(), num_experts=5)
entropy_id = (-(final_probs * final_probs.clamp_min(1e-9).log()).sum(dim=1)).detach().cpu().numpy()

# OOD AUROC por entropía si existen arrays OOD de CLS
ood_auroc = None
ood_hint = 'No disponible (faltan arrays OOD en este notebook)'
if 'Z_ood_np' in globals():
    Z_ood_t = torch.tensor(Z_ood_np, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        ood_probs = F.softmax(router.router_head(Z_ood_t), dim=-1)
    entropy_ood = (-(ood_probs * ood_probs.clamp_min(1e-9).log()).sum(dim=1)).detach().cpu().numpy()
    y_bin = np.concatenate([np.zeros_like(entropy_id), np.ones_like(entropy_ood)])
    s_bin = np.concatenate([entropy_id, entropy_ood])
    ood_auroc = float(roc_auc_score(y_bin, s_bin))
    ood_hint = 'Calculado con entropía del gating'

# VRAM pico aproximada de entrenamiento (si hay history)
peak_vram_mb = max([h.get('vram_mb', 0.0) for h in history_router], default=0.0)

f_final = torch.bincount(final_preds.detach().cpu(), minlength=5).float()
f_frac = (f_final / (f_final.sum() + 1e-9)).numpy()

print('=' * 80)
print('EVALUACION FINAL DEL ROUTER (VALIDACION)')
print('=' * 80)
print(f'Routing Accuracy: {final_acc:.4f}  | objetivo > 0.80')
print(f'Load Balance ratio max/min: {ratio_final:.3f}  | objetivo < 1.30')
print(f'VRAM pico (MB): {peak_vram_mb:.1f}  | objetivo < 11776 MB (~11.5 GB)')
print(f'OOD AUROC: {ood_auroc if ood_auroc is not None else "N/A"}  | objetivo > 0.80')
print(f'Nota OOD: {ood_hint}')

print('\nDistribucion de carga f_i:')
expert_names = ['NIH (Exp1)', 'ISIC (Exp2)', 'Osteo (Exp3)', 'LUNA16 (Exp4)', 'Pancreas (Exp5)']
for name, frac in zip(expert_names, f_frac):
    print(f'  {name}: {100.0 * frac:.2f}%')

print('\nChecklist KPI:')
print(f"  [ {'OK' if final_acc > 0.80 else 'FAIL'} ] Routing Accuracy > 0.80")
print(f"  [ {'OK' if ratio_final < 1.30 else 'FAIL'} ] Load Balance ratio < 1.30")
print(f"  [ {'OK' if peak_vram_mb < 11776 else 'FAIL'} ] VRAM < 11.5 GB")
if ood_auroc is not None:
    print(f"  [ {'OK' if ood_auroc > 0.80 else 'FAIL'} ] OOD AUROC > 0.80")
else:
    print('  [ PENDING ] OOD AUROC > 0.80 (requiere Z_ood_np)')
print('=' * 80)


## Curvas de Entrenamiento — Mecanismo A (Router ViT+Linear)

Visualización del Routing Accuracy durante el entrenamiento del Mecanismo A sobre
los CLS tokens. Corresponde a la **Figura 3** del reporte técnico (curva del ablation study).

In [ ]:
import matplotlib.pyplot as plt

# Curvas entrenamiento final
epochs_list = [h['epoch'] for h in history_router]
train_accs = [h['train_acc'] for h in history_router]
val_accs = [h['val_acc'] for h in history_router]
train_loss = [h['train_loss'] for h in history_router]
val_ratio = [h['val_ratio'] for h in history_router]
val_entropy = [h['val_entropy'] for h in history_router]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Router ViT+Linear - Warmup + Partial Unfreeze', fontsize=13)

axes[0].plot(epochs_list, train_accs, label='Train Acc', color='steelblue')
axes[0].plot(epochs_list, val_accs, label='Val Acc', color='darkorange')
axes[0].axhline(0.80, ls='--', color='red', alpha=0.6, label='Meta 0.80')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Routing Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_list, train_loss, color='purple', label='Train Loss')
axes[1].axhline(1.30, ls='--', color='red', alpha=0.6, label='Ratio limite')
axes[1].plot(epochs_list, val_ratio, color='teal', label='Val Ratio max/min')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Valor')
axes[1].set_title('Loss + Balance Ratio')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs_list, val_entropy, color='brown')
axes[2].axhline(np.log(5), ls='--', color='gray', alpha=0.6, label='log(5)')
axes[2].set_xlabel('Epoca')
axes[2].set_ylabel('Entropy')
axes[2].set_title('Entropia media de gating')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(FEATURE_DIR, 'router_training_curves.png')
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()

# Curva comparativa de warmup por alpha
plt.figure(figsize=(8, 5))
for alpha, hist in warmup_histories.items():
    xs = [h['epoch'] for h in hist]
    ys = [h['val_acc'] for h in hist]
    plt.plot(xs, ys, label=f'alpha={alpha:.2f}')
plt.axhline(0.80, ls='--', color='red', alpha=0.6)
plt.xlabel('Epoca')
plt.ylabel('Val Routing Accuracy')
plt.title('Warmup sweep por alpha_aux')
plt.grid(alpha=0.3)
plt.legend()
warmup_path = os.path.join(FEATURE_DIR, 'router_warmup_sweep.png')
plt.savefig(warmup_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Curvas guardadas en: {curve_path}')
print(f'Sweep warmup guardado en: {warmup_path}')


## Resumen: Artefactos Generados

Al completar este notebook se habrán producido los siguientes archivos en Drive:

| Archivo | Descripción |
|---------|-------------|
| `ablation_data_v1/Z_train.npy` | CLS tokens de entrenamiento `[N_train, 192]` |
| `ablation_data_v1/Z_val.npy` | CLS tokens de validación `[N_val, 192]` |
| `ablation_data_v1/y_train_expert.npy` | Etiquetas de experto para train |
| `ablation_data_v1/y_val_expert.npy` | Etiquetas de experto para val |
| `ablation_results.csv` | Tabla comparativa de los 4 mecanismos |
| `router_head_phase2.pth` | Checkpoint del router head entrenado |
| `router_training_curves.png` | Figura 3 del reporte técnico |

**Siguiente paso:** Notebook `04_Entrenamiento_MoE_Fases.ipynb` —
Integración del router ganador con los 5 expertos congelados para el sistema MoE completo,
incluyendo la `L_task` sobre las etiquetas de enfermedad (no solo de routing).